# Contribute to KataGo training on Google Colab

Use the **control panel** below. You can choose settings, start the server, copy its connection address and stop it without editing code.

During setup, the panel shows the current step, elapsed time and three recent log lines. If a step is quiet, it shows the time since the last log. Full logs remain below.

## Open the panel

1. Choose **Runtime > Change runtime type > T4 GPU**. GPU availability depends on your plan and usage. Start with the latest runtime version.
2. Click **▶ on the Open control panel block directly below this guide**. Wait for the panel to appear in that block. This is the only code block you need to run. It prepares the controls without logging in, starting a server, deploying an app or running Stop.
3. Models are selected by the training server. Click **Start server** and allow Google Drive access when prompted. The first start can take several minutes.
4. Copy the complete address in **SWHub URL** into **Tools > KataGo Contribution > Settings**. Enter your verified [katagotraining.org](https://katagotraining.org/accounts/signup/) account in SWHub, then select **Start contributing**.
5. Select **Finish games & stop** in SWHub and wait for the games to upload. Then click **Stop server** in this panel. Force stop and disconnects can abandon unfinished games.

Keep the connection address private. Keep Colab connected while using the server. Temporary tunnel addresses can change after restarting.

## Runtime and saved files

Colab does not guarantee a GPU or an uninterrupted session. Contribution requires an eligible paid runtime with a positive compute-unit balance. Free managed runtimes restrict distributed computing workers. [Read Colab’s rules](https://research.google.com/colaboratory/faq.html).

Completed downloads and contribution data use `MyDrive/SWHub/KataGoContribution`. Data is saved every ten minutes and when stopped. Contribution sessions last up to four hours, including paused time. KataGo runs from runtime-local `/content`. Passwords and incomplete downloads stay out of the Drive cache.

All code blocks are collapsed by default. Expand them only if you want to inspect the implementation. If controls do not appear, reconnect and run the top Open control panel block again. Trust the notebook in local Jupyter. After changing settings, stop an active Colab server first, then start again. Modal changes take effect after Deploy.


In [ ]:
# @title ▶ Open control panel
# Generated by tool/katago_notebooks/sync_controls.py.
# Implementation sources remain readable in the collapsed cells below.
%pip install -q "ipywidgets>=7.7.1,<9"

try:
    from google.colab import output
except ImportError:
    pass
else:
    output.enable_custom_widget_manager()

"""Shared controls embedded in all four standalone KataGo notebooks.

Run sync_controls.py after editing. No repository checkout is needed by users.
"""

import ast
import builtins
from collections import deque
import html
import json
import re
import subprocess
import threading
import time
from pathlib import Path

import ipywidgets as widgets
from IPython import get_ipython
from IPython.display import display


# Registering a cell saves its source without running setup or starting a GPU.
SWHUB_ACTIONS = {}


def swhub_action(line, cell):
    SWHUB_ACTIONS[line.strip()] = cell


get_ipython().register_magic_function(swhub_action, "cell", "swhub_action")


def swhub_configure_source(source, settings):
    """Replace only top-level setting assignments, preserving embedded servers."""
    tree = ast.parse(source)
    lines = source.splitlines(keepends=True)
    for node in reversed(tree.body):
        if not isinstance(node, ast.Assign) or len(node.targets) != 1:
            continue
        target = node.targets[0]
        if isinstance(target, ast.Name) and target.id in settings:
            lines[node.lineno - 1:node.end_lineno] = [
                f"{target.id} = {settings[target.id]!r}\n"
            ]
    return "".join(lines)


def swhub_run_action(name, settings=None, on_output=None):
    source = SWHUB_ACTIONS[name]
    # Existing writefile cells remain readable, self-contained service sources.
    lines = source.splitlines(keepends=True)
    writefile = next((i for i, line in enumerate(lines)
                      if line.startswith("%%writefile ")), None)
    if writefile is not None:
        path = Path(lines[writefile].split(maxsplit=1)[1].strip())
        source = swhub_configure_source("".join(lines[writefile + 1:]), settings or {})
        compile(source, str(path), "exec")
        path.write_text(source, encoding="utf-8")
        print("Prepared", path.name)
        if on_output is not None:
            on_output("Prepared " + path.name)
        return
    source = swhub_configure_source(source, settings or {})
    # Observe Python action logs without redirecting the kernel's stdout or
    # moving Drive authorization and IPython work off the calling thread.
    had_print = "print" in globals()
    original_print = globals().get("print", builtins.print)

    def report(*args, **kwargs):
        original_print(*args, **kwargs)
        if on_output is not None and kwargs.get("file") is None:
            separator = kwargs.get("sep", " ")
            on_output((" " if separator is None else separator).join(str(arg) for arg in args))

    globals()["print"] = report
    try:
        exec(compile(source, f"KataGo action: {name}", "exec"), globals())
    finally:
        if had_print:
            globals()["print"] = original_print
        else:
            globals().pop("print", None)


class SWHubKataGoPanel:
    def __init__(self, kind):
        self.kind = kind
        self.modal = kind.startswith("modal")
        self.contribution = kind.endswith("contribution")
        self.busy = False
        self.target = None
        self.activity_stop = threading.Event()
        self.activity_lock = threading.RLock()
        self.recent_activity = deque(maxlen=3)
        self.activity_secrets = ()
        self.buttons = {}
        self.fields = {}
        self.status = widgets.HTML(value="Ready. Choose your settings, then start below.")
        self.activity = widgets.HTML(layout=widgets.Layout(max_height="120px", overflow="auto"))
        self.endpoint = widgets.Text(description="SWHub URL", placeholder="Address appears after starting",
                                     layout=widgets.Layout(width="100%"))
        self.connection = widgets.HTML()
        self.output = widgets.Output(layout=widgets.Layout(
            max_height="180px", overflow="auto", border="1px solid #888"))
        self.progress = widgets.IntProgress(min=0, max=1, description="Progress")
        children = [widgets.HTML(value="<h2>SWHub · KataGo " +
                    ("Contribution" if self.contribution else "Analysis") + "</h2>")]
        if self.modal:
            children += [widgets.HTML(value="Modal supplies the T4 GPU. GPU use requires a valid "
                         "payment method. Closing this notebook does not stop the deployed app."),
                         self.button("login", "1. Login", self.login),
                         self.field("MODAL_PROFILE", widgets.Dropdown, "Workspace", options=[]),
                         self.field("MODAL_ENVIRONMENT", widgets.Text, "Environment", value="main"),
                         self.button("profiles", "Refresh workspaces", self.refresh_profiles),
                         self.button("select", "2. Select workspace", self.select_workspace)]
            self.fields["MODAL_PROFILE"].observe(self.invalidate_target, names="value")
            self.fields["MODAL_ENVIRONMENT"].observe(self.invalidate_target, names="value")
        if not self.contribution:
            children += [self.field("MAIN_MODEL_PRESET", widgets.Dropdown, "Model", options=[
                ("Medium transformer (recommended)", "transformer_medium"),
                ("Small transformer", "transformer_small"),
                ("Large transformer", "transformer_large"),
                ("Convolutional b18", "colab_fast_b18"),
                ("Convolutional b28", "latest_b28"),
                ("Convolutional b40 Zhizi", "strongest_b40_zhizi"),
                ("Custom model", "custom")]),
                self.field("ENABLE_HUMAN_MODEL", widgets.Checkbox, "HumanSL", value=True),
                widgets.HTML(value="Keep HumanSL enabled for Career, Subjects and Game Focus.")]
            custom = [self.field(key, widgets.Text, label) for key, label in (
                ("MODEL_URL", "Model URL"), ("MODEL_FILENAME", "Filename"),
                ("MODEL_SHA256", "SHA-256"))]
            advanced = widgets.Accordion(children=[widgets.VBox(custom)], selected_index=None)
            advanced.set_title(0, "Custom model settings (only for Custom model)")
            children.append(advanced)
        else:
            children.append(widgets.HTML(value="The training server chooses the model. Enter your "
                            "katagotraining.org account in SWHub. Finish games & stop in SWHub "
                            "before stopping this server. Sessions last up to four hours."))
        if self.kind == "colab_analysis":
            children += [self.field("TUNNEL_PROVIDER", widgets.Dropdown, "Connection", options=[
                ("Automatic (recommended)", "auto"), ("Cloudflare", "cloudflare"),
                ("Pinggy", "pinggy"), ("localhost.run", "localhostrun"),
                ("Local only", "none")]),
                self.field("USE_GOOGLE_DRIVE_CACHE", widgets.Checkbox, "Drive cache", value=True)]
        if self.kind == "modal_contribution":
            children += [self.field("ENABLE_CONTRIBUTOR_LOGIN", widgets.Checkbox,
                                    "Controller login", value=False),
                         self.field("CONTRIBUTOR_LOGIN_USERNAME", widgets.Text, "Username"),
                         self.field("CONTRIBUTOR_LOGIN_PASSWORD", widgets.Password, "Password"),
                         widgets.HTML(value="Optional extra server login, separate from your training account. "
                                      "Enter it again for each deployment. Keep the server URL private.")]
        children += [self.button("start", "3. Deploy" if self.modal else "Start server",
                                 self.start, "success"), self.progress, self.status, self.activity,
                     widgets.HTML(value="<b>Connect SWHub</b>. Copy the full address below into " +
                                  ("Tools → KataGo Contribution → Settings." if self.contribution else
                                   "Tools → KataGo → Settings → Remote, then select Save.")),
                     self.endpoint, self.connection,
                     self.button("logs", "Show recent logs", self.logs),
                     self.button("stop", "Stop Modal app" if self.modal else "Stop server",
                                 self.stop, "danger")]
        children += [widgets.HTML(value="<b>Full setup and troubleshooting logs</b>. "
                                  "Wait for an action to finish before clicking another. "
                                  "Use the notebook's Interrupt execution control if setup stalls."), self.output]
        self.view = widgets.VBox(children, layout=widgets.Layout(max_width="850px"))
        self.update_buttons()

    def field(self, key, widget_type, description, **kwargs):
        field = widget_type(description=description, style={"description_width": "120px"},
                            layout=widgets.Layout(width="100%"), **kwargs)
        self.fields[key] = field
        return field

    def button(self, key, description, action, style=""):
        button = widgets.Button(description=description, button_style=style,
                                layout=widgets.Layout(width="230px"))
        button.on_click(lambda _: self.perform(description, action))
        self.buttons[key] = button
        return button

    def update_buttons(self):
        for button in self.buttons.values():
            button.disabled = self.busy
        for field in self.fields.values():
            field.disabled = self.busy
        if self.modal and not self.busy:
            for key in ("start", "logs", "stop"):
                self.buttons[key].disabled = self.target is None
            self.buttons["select"].disabled = not self.fields["MODAL_PROFILE"].value

    def invalidate_target(self, change=None):
        self.target = None
        globals()["selected_modal_target"] = None
        self.endpoint.value = ""
        self.connection.value = ""
        self.status.value = "Select workspace to verify the new deployment target."
        self.update_buttons()

    def perform(self, label, action):
        if self.busy:
            return
        self.busy = True
        self.update_buttons()
        self.status.value = html.escape(label) + "…"
        self.progress.bar_style = "info"
        self.progress.max = 1
        self.progress.value = 0
        self.started_at = self.last_activity_at = time.monotonic()
        self.last_render_at = 0
        self.recent_activity.clear()
        secret = self.fields.get("CONTRIBUTOR_LOGIN_PASSWORD")
        self.activity_secrets = (secret.value,) if secret is not None and secret.value else ()
        self.activity_stop = threading.Event()
        self.render_activity()
        self.activity_thread = threading.Thread(target=self.watch_activity, daemon=True)
        self.activity_thread.start()
        with self.output:
            print("\n" + label)
            try:
                action()
            except (Exception, KeyboardInterrupt) as exc:
                self.progress.bar_style = "danger"
                self.status.value = "Action interrupted." if isinstance(exc, KeyboardInterrupt) else (
                    "Action failed. Read the details below, correct the problem and retry.")
                # Passwords are never printed in tracebacks or saved to notebook source.
                message = str(exc)
                for secret in self.activity_secrets:
                    message = message.replace(secret, "[redacted]")
                print(type(exc).__name__ + ": " + message)
            else:
                self.progress.bar_style = "success"
                self.progress.value = self.progress.max
                self.status.value = ("Server address ready. Connect SWHub below." if action == self.start
                                     else html.escape(label) + " complete.")
            finally:
                self.activity_stop.set()
                self.activity_thread.join(timeout=2)
                self.render_activity(finished=True)
                if action == self.start:
                    self.clear_controller_password()
                self.activity_secrets = ()
                self.busy = False
                self.update_buttons()

    def run_action(self, name, settings=None):
        swhub_run_action(name, settings, on_output=self.record_activity)

    def record_activity(self, message):
        with self.activity_lock:
            if not self.busy or self.activity_stop.is_set():
                return
            # Keep the preview bounded, strip terminal escapes, and escape HTML.
            message = re.sub(r"\x1b\[[0-?]*[ -/]*[@-~]", "", message)
            for secret in self.activity_secrets:
                message = message.replace(secret, "[redacted]")
            for line in message.replace("\r", "\n").splitlines()[-3:]:
                line = " ".join(line.split())
                if not line:
                    continue
                self.last_activity_at = time.monotonic()
                elapsed = int(self.last_activity_at - self.started_at)
                self.recent_activity.append((elapsed, line[:180]))
                stage = re.match(r"\[(\d+)/(\d+)\]", line)
                if stage:
                    self.status.value = html.escape(line[:180])
                    self.progress.max = max(1, int(stage[2]))
                    self.progress.value = min(self.progress.max, max(0, int(stage[1]) - 1))
            if time.monotonic() - self.last_render_at >= 0.2:
                self.render_activity()

    def render_activity(self, finished=False):
        with self.activity_lock:
            now = time.monotonic()
            self.last_render_at = now
            elapsed = int(now - self.started_at)
            quiet = int(now - self.last_activity_at)
            timing = f"Elapsed {elapsed // 60}:{elapsed % 60:02d}"
            if not finished:
                timing += (f" · No new log for {quiet}s. Waiting for this step to finish."
                           if quiet >= 10 else " · Working")
            lines = [f"<div><small>{timing}</small></div>"]
            for stamp, line in self.recent_activity:
                lines.append(f"<div>{stamp // 60}:{stamp % 60:02d} · {html.escape(line)}</div>")
            self.activity.value = '<div style="overflow-wrap:anywhere">' + "".join(lines) + "</div>"

    def watch_activity(self):
        while not self.activity_stop.wait(1):
            self.render_activity()

    def refresh_profiles(self):
        self.invalidate_target()
        # Reject token overrides before interpreting CLI output as profiles.
        import os
        if os.environ.get("MODAL_TOKEN_ID") or os.environ.get("MODAL_TOKEN_SECRET"):
            raise RuntimeError("Token environment variables override workspace profiles. "
                               "Remove them or use a clean runtime, then Login again.")
        result = subprocess.run(["modal", "profile", "list", "--json"],
                                check=True, text=True, capture_output=True, timeout=60)
        profiles = json.loads(result.stdout)
        names = [row["name"] for row in profiles]
        self.fields["MODAL_PROFILE"].options = names
        print("Choose a workspace and select it." if names else "No saved workspaces. Login first.")

    def login(self):
        self.invalidate_target()
        self.run_action("login")
        self.refresh_profiles()

    def select_workspace(self):
        self.invalidate_target()
        self.run_action("workspace", {key: self.fields[key].value for key in (
            "MODAL_PROFILE", "MODAL_ENVIRONMENT")})
        self.target = globals()["selected_modal_target"]

    def settings(self):
        values = {key: field.value for key, field in self.fields.items()}
        if not self.contribution:
            values["HUMAN_MODEL_PRESET"] = (
                "official_human_sl_v0" if values["ENABLE_HUMAN_MODEL"] else "disabled")
            if values["MAIN_MODEL_PRESET"] == "custom":
                from urllib.parse import urlparse
                import re
                if urlparse(values["MODEL_URL"]).scheme != "https":
                    raise ValueError("Enter an HTTPS URL for the custom model.")
                filename = values["MODEL_FILENAME"]
                if not filename or filename in (".", "..") or "/" in filename or "\\" in filename:
                    raise ValueError("Enter a model filename without folders.")
                if re.fullmatch(r"[0-9a-fA-F]{64}", values["MODEL_SHA256"]) is None:
                    raise ValueError("Enter the custom model's 64-character SHA-256.")
            else:
                for key in ("MODEL_URL", "MODEL_FILENAME", "MODEL_SHA256"):
                    values.pop(key)
        if self.modal:
            values["KEEP_STREAMING_AFTER_READY"] = False
        return values

    def start(self):
        if not self.modal:
            for name in (("service", "tunnel") if self.contribution else ("ws_proc", "tunnel_proc")):
                process = globals().get(name)
                if process is not None and process.poll() is None:
                    raise RuntimeError("Server is already running. Stop server before starting again.")
        self.endpoint.value = ""
        self.connection.value = ""
        for key in ("final_wss_url", "final_status_url", "websocket_url", "base", "ready"):
            globals().pop(key, None)
        settings = self.settings()
        if self.modal and (self.target is None or self.target != globals().get("selected_modal_target")):
            self.invalidate_target()
            raise RuntimeError("Select workspace again before deploying.")
        if self.kind == "modal_contribution":
            if settings["ENABLE_CONTRIBUTOR_LOGIN"] and not all(settings[key].strip() for key in (
                    "CONTRIBUTOR_LOGIN_USERNAME", "CONTRIBUTOR_LOGIN_PASSWORD")):
                raise ValueError("Enter both controller login fields or disable Controller login.")
            globals().update({key: settings[key] for key in (
                "ENABLE_CONTRIBUTOR_LOGIN", "CONTRIBUTOR_LOGIN_USERNAME", "CONTRIBUTOR_LOGIN_PASSWORD")})
        steps = (["write_app", "deploy"] if self.modal else
                 ["prepare", "drive", "write_app", "start"] if self.contribution else ["start"])
        self.progress.max = len(steps)
        self.progress.value = 0
        try:
            for index, step in enumerate(steps):
                stage = {"prepare": "Checking GPU and installing dependencies",
                         "drive": "Connecting Google Drive and restoring cached files",
                         "write_app": "Preparing the server",
                         "deploy": "Deploying to Modal, waiting for startup",
                         "start": "Preparing KataGo and opening the connection"}[step]
                self.status.value = f"Step {index + 1}/{len(steps)}: {stage}…"
                self.record_activity(stage)
                self.run_action(step, settings)
                if self.progress.max == len(steps):
                    self.progress.value = index + 1
            if self.kind == "colab_analysis":
                url = globals().get("final_wss_url", "")
                status_url = globals().get("final_status_url", "")
            elif self.kind == "modal_contribution":
                url = globals().get("websocket_url", "")
                status_url = ""
            else:
                base = globals().get("base", "")
                url = base.replace("https://", "wss://").replace("http://", "ws://") + (
                    "/contribute" if self.contribution else "/katago") if base else ""
                status_url = base + ("/" if self.contribution else "/status") if base else ""
            if not url:
                raise RuntimeError("No connection URL found. Check the deployment logs below.")
            self.endpoint.value = url
            if status_url:
                self.connection.value = '<a target="_blank" rel="noopener" href="' + html.escape(
                    status_url, quote=True) + '">Open server status</a>'
            if self.kind == "modal_analysis" and not globals().get("ready"):
                raise RuntimeError("Deployment finished, but KataGo readiness was not confirmed. "
                                   "Check recent logs or open server status before connecting.")
            print("Connection address ready. Keep it private.")
        finally:
            self.clear_controller_password()

    def clear_controller_password(self):
        if self.kind == "modal_contribution":
            self.fields["CONTRIBUTOR_LOGIN_PASSWORD"].value = ""
            # The existing deployment cell keeps its subprocess environment in globals.
            for key in ("CONTRIBUTOR_LOGIN_PASSWORD", "password"):
                globals().pop(key, None)
            environment = globals().get("env", {})
            environment.pop("SWHUB_CONTRIBUTOR_PASSWORD", None)
            environment.pop("SWHUB_CONTRIBUTOR_USERNAME", None)

    def logs(self):
        if self.modal:
            if self.target is None or self.target != globals().get("selected_modal_target"):
                self.invalidate_target()
                raise RuntimeError("Select workspace again before viewing logs.")
            app = "Katago-Contribute" if self.contribution else "KataGo-Analyse"
            # Recent logs return promptly, the deploy action already streams startup.
            subprocess.run(modal_command("app", "logs", app), check=True, timeout=30)
        else:
            paths = (["/content/colab_katago_contribute.log", "/content/colab_cloudflared.log"]
                     if self.contribution else ["/content/katago_ws_server.log", "/content/cloudflared.log"])
            for filename in paths:
                path = Path(filename)
                if path.exists():
                    print(path.name + "\n" + path.read_text(errors="replace")[-12000:])

    def stop(self):
        if self.modal:
            if self.target is None or self.target != globals().get("selected_modal_target"):
                self.invalidate_target()
                raise RuntimeError("Select workspace again before stopping.")
        self.run_action("stop")
        self.endpoint.value = ""
        self.connection.value = ""


def show_swhub_panel(kind):
    previous = globals().get("swhub_panel")
    if previous is not None:
        previous.activity_stop.set()
        for button in previous.buttons.values():
            button.disabled = True
        previous.view.close()
    panel = SWHubKataGoPanel(kind)
    globals()["swhub_panel"] = panel
    display(panel.view)
    return panel


# Load the bundled actions without executing setup or starting a server.
import base64 as _swhub_base64
import zlib as _swhub_zlib
_swhub_action_bundle = 'eNrtfQtb40iS4F/R0DeHXWUL24B57HpuaKC6mK6uYoHqvt0yq0+W0liNLGkkGYpm+O8XEZkpZUop2dT07Oy333XPNJaUz8jIeGfk81aSssRN2daxtfWd9ec8yENmXbG7IMtZaomPViddRZkVR+GTNU/jpZUvmJW4EQu706io9bMbBr6bM+v089mJ9cPlZ8uNfCuIstwNQ+s0Dt2Z5bOERT6LvIBl0yhYJnGaWws3W4TBrGfFWc/KFqs8COHvapaksccyfPcE/1mlIRSyU/bXFcvyaUQDSdwcq1qipUt4nEbQ8Fy0Yz8uAm/RmW5FD4EfuP1sGUy3ulaQWR/jiB1PIwv+Sd0gg0mvojxYsvM0jdMO/4D/TLduFlA8inM2i+N7C/sPUpZZrpjSx58vzi74fFPehA2V1AZOF3FcdmD9yTpduNEdk8Wt/CnBtzd7BdBSBt9s2UoXp3SXrBwAyIxZEwU2NpQTg/2iT7IHHff7AKv0qQ91J5G7ZPLtPE6Xbj7xsodeFC+Y67N0unXb4+3k7Gs+uUlXrGd5bpKvUubEqzxZFS8XzLufvHPDjEGNLgG7GBwsD9SIvNhn1h8m1sCKUwSdUiDLfWgO/qRB0un+c1agb16CfvsaJGkQ5YBKJ1kW3EXMV/o9Rsg2zdHOkhA2SBCxrNP9Mril5fTZXG4NR90UnSDywpXPHDdyw6csyDikJaDkGC541SC6E3vRu3fvACQRYz6MbPYErxWY2TZMpKs3cfmUL+KIRg77y35gaRbEER+sHCfhQ/p0XMKygnqAdIAj/TuWc9xaJUgBBJ799a+IVQJhEHm6ZTvfWZ9nAPuVNdqzB3vWKoPB40zy8Z6FqNorvo+K77DRfwuSPbtsBR7bN4X8R4zTc2EwfHTZIn7kv0Sz0HO5CeQ/ymbgizpRurm8uDyn9yxN1fdn5z9//PzhQ6Upfd/It93KZPhCwnT0cVmwyYrJqptsgpsM0bX8qmOfxaDDsjV1X2wCsWJlBbKKpX1Slhh/eG7fY2kezAMP1j8TL4Fkwy+9XXj/WDSaHMCY+vOVbDfNniJPYFIEn4oFuouXybB4goe+zx5MbcewlbJs0ffCgEXUiwLWW+OS0PIaV6TYVhMEx9wFCCTBZDKwh+ORPRLjfAi8OI3w7T6gMn/5yGZZ7N2zPJtMhgc2IlbZaoA7X9/kx/q4ZL+2myBlQPYFLM9xETpBDE3aY+ypu+lS4v5mX5m3yt1ZyGjhlmIBJIz15e37QYZF+/C9LyhDn6Al1x3/vpHjfA1c2VePJbk64FPol/mX/ImIv+VmWE6BSiuD4Ct/KbaOmImbw5ituRtA2zZINK5P9MWLl0vcLoz3M4sfmKD2QOes69wFKSJjKcxZ5yG8j4s5J7ewz0K2hK0PxYHcunMUliQ7kjyF00IgEIxRLUmMtzPrJo1X0ALQoDhHGp4xD4drqz12uagFcEB+AWzWxC2Il5x++PT57N2Hk6vzM+fn86vri08fiYCMBqOxPRzYB9isWujz1QcoMJ9uLfI8yY53du6CfLGa2QCcHS+MV/48BKlP+envpCxkLhDhHT9+jMLY9XeeDd2+qHX6wJ5WX/vu0h/vVUdw/f5ktD+mUQ7nR0fz2f7wcHa46zNvNHMP/cODgXvgeUdHQ/9w5B4csPHwaDw62nd3/dHB/t7MHR64s6P50fDwYIRNK706KBJCwygKwsbZ8eIoB0qgDox2jlolXyZQo9qI/QhQcbLVfB58hZZsKEU1OfOeA2I52cKFaXSwtOTQfgAbIofmhFRrizIC/bFNklptpFTQbDpDgRQQPotXqccUnAcpDXbSKroHlLYCwLBO6C5nvnssigIbcP3OcDDas95Y+Kfbs6AxaK5CTviQbI6PHWpSjIYzEllgwb7yXxytgE6h6FYDC/sKygGgHsp2KhSqBbsoAdYXXQyuAn97FQG63HeWAUhX0Z0TaxKDEPoBqroWYF/xv50KdvcsLtVmk+fp1mfYz/2TO2IHx4Bw17+8X836tFd3hkhHX4ySDq1UpTt4pGUTzz0LNzpKBcPRgBYRxNIkjjIUkyvzE8v9KJeby9SVlRJKixcnTwjZePZrp2yR11CIvuvlKzfEndywCtBvV+M7okbburRQXCAYSusWVOvjJoYlA23CWxxbz7z9F401xcCSWBK6HquOrVfDLZ1HnNOfAIXUorVXoU0xjzqFsL3FMvY7g/hgfx9KCx4nNJ26NPkFpKk6gnNeKeqSrLuZ8kRDFHIa7TMTBce9BxtfH1mzvgSLc/41AT4CK6OukpFMg7IS57BeeuN83bh+oH+RwiR8Br6/5afBQ4PFgD5tZi84jaMIxmv9EMd38HhGNTkvzvI45UwzwXFkSMEtLrxzvf+O6sBGQYYrlH/qus0uIB7KpUUqByONEHOQ6QceSt53YTwDKR0UN5BTYcuKLwQcWKtqBa5mkT2Bj17/bidxGHY2MTlsXedxQpMWTVjIu9JgtgJoWDM2R5hw2ODHIM84SGzJlXD+9jJekXpXMD4OFixCvxyqY2CQ9HXnpydahx0ikjs/urn7Q3wqh0FY3iWY4bgbm7qHandxvxh+rXt7ee8HaQdtSxHIyHxvEF9RdrDWy0Y1OGdGKdnRKjt57FDvhcGhUV7+UhX5pEpSfd93TS9BLQlZzvCP+9RWAEgcagC+sYz4OHnDZY51Jew3tTJADZ41ELzsGMsoa1Ipocr0NXmeYN1ghYraYFeDWwmShtdmSLVDaR2E2mfeBjsJFQ0iim3oipMuX6VbfBPxvUrkkGwuSv+Cqj6mIOQ5oPKZKSt9tuDzZtT1FyqOn8LYA57vKZvYkiQNipNtzWNhyDtAq1rGwjltXjdAM1dB6KI8pgY1NceeRn/8I1VFKcQq5W0s5HBK4JSUwE6eAHe28F8Bl4ehPTwEETQTJfx+DtSxRv9QGiaiBBPnI7ARCWn4coQEFyATwFUANsA8qyoZGaRRt0zijKFCCHrY3cJyQTkpWKb1b6vAu7dugAaz0LZ+ZIwTZRAgWeoie83pk4VaFKz7A7yz5ZxIbuZsRmjrVSt3+bx0veLh1ywua8alaTxlJdsi2dDIxeSrp/J3zpYJLkjxomo+F69/C0SxZrZJX0R1oL/At8om6Znrp4hMAli0JBoOLVgIO8IKQG/NA0RHWJxZDAI2Xx+pGmc2tkL+g+FunzA8WJIFxo+Jy8KUH6ARjoZUbGRdf/p48tN5hjZCsnxyvIKWsIBz9fnjzcVP587ZxZXkVNsFkmaPoPb2Jbta+e5w1B+O7EP8HUVH9uF2VzZz/m+fL67Ofzr/eHMN7Wxvb0+n00gY3LFmX+AYPAxHkwm2Yh8NrH4f133ChfNj1wet5ND3Zv6A+fO90cHsYDTY8/cOB/54MB77u7tHg5Hnzkbu3sHgwPfm3t6hO/IHc39vDxTegdLjLHQzta89e1jp7ND19tjBwdAHtXnv0NvfH2K7ozHzXVCnd73hkeuOdwej3fFsb3g4OBwPRgN/vj9zj8bjg+HeSJ1eFIm+ACb2wD48qPTlj2cwcJjVLhvtue7u0B9AF+6ht3s0B51+92h3f293PPAP/MPhYH+f+b4/Gx/Nd/f9/V13zzvUQRk9pLmnAXK3CsiDg/3x/ujwcH/E5oeHRwcHI/dwzxt7/uGRvzf05+7BHswCBuN57nA8m+0O54fzwXgIAzw6Otor+osefiXb/H1rd4fD+Xy8u3swdA/YzB+z8d7+8OhgMD7cmx9B62zE3NngYHc0nB3sekf+aG/P90azPd89mB/szg5heogwkZRQkB7OgzvUDnC+Duys1E2fSKnoSNaTxnE+QZFRyizfWSdWxB6B9wShb4nNby1XWS5NTBmjHZyhnuFGd9I7UBqeZEMMZp/G0RLZkx8zLsF63CXCWQZqz7wxK56TrCmFUu43EEZ4dbCwLWo7jsvLZREhBXOjtPpFGk+QdoN8y22uqHSRsgZSbxyi+NYlPkAmHqAgWYzmNw1iNkrvnW2+ujtvdgC0293uba15GxQ1NK0maPDjbaakb4G6KqBD4v/2hzPnw8X3VydX/+5cnty83+7B3hcunQ6UJRCxpIszxSaE5oktAnBT4kJofsVyx5oujiCnIhXFG2lTEK2YosJy4BLtohrdipEoBnoc4VpCZ51tThtwoIRcAjriOYqKD7TL8KnYA9tVyxG2LoFGU+nQWHYsAeBt+Fn2Dq8R3D1LFKp9qTYvIFH0YAeZg1K+qVx1/YRpHFGkrF/iSQVE6yaxjbt/uzaDbajiLbabxr/p2F8/boHlIPdFclGBLg5HhNDjPbmG9CZ3U8DUbOfr4dgZ73Gr6w6fBhUSVbrHLSARO0mwx1WW7pDUCJhO+0kMpftfBwClZah0rJmTxP78Utubt7BLyj1p/xoHEfTh5TaKMPfsKesobXa7CkEmz5mj7peOG4bxo0Nb5lC47awlyKXL1ZIKos3vcFAS50vyQyL1zHKQM90Q6ZwgpCSKzmGSC6sIaojYA3y5IQTLpCAZPsnmkpRRZd/i3AFBZ11wv5Hlrc4+foRaoftbADKSKFjut7KK9PlyH2lKbEiIbx46vWEgKHv2uPCI80JLOH2xT88+fOgg4uPrNLez2IZNUlrLypKeg+oPt49gYRv/I6wbP7D8Z16+I0rPnlI2lxamblcLFMB2kF4CTj98Gd12rX+dSBOY/eCGK2b9K4iGg8Gg2ZCyfUKQcPNgFpZSolQBgkwGEPg4FdwJ6G/mu4zmitRTzLVnla8+5NpLop6VN38J8g9ARiWgpKVbgSZ2Bb2S94lY3C0fwtL9NSY06Xw56lmHt4j/BRiGt+jm3R5uc7b55ehWNq1zD8LVyvLNxfpFEY7qmfp52e5WKtn033KlcD9SSISywlnwG3PysiJfjYm5gY7mRU1znGuH19jZsYawfoOeaOGP/FG8194SV+XPf5qIYgQCvSm9JWNDGuelAX0Z3CLiccAL/p8BcnU6uAA9i3fOPwP0j0TH8AW+H9Qoockiuw3btURE2rMSl61n6k9fCbl9CVjbccJ5OOfYrv+Af+5SN1ngjwVbpaAyB8TAGcp6LAMqxrARdHmqrwtjhvxmnhr2CPt4zlLRb/kAvYuHNtGjnEBVpDEipPNcVHhpQE5uVkECZfurZZKRuJapzH2WMvde8xp0Pl0T/HvaapDPhbadMja+DVWORG+wfQRJC4n5F2ubsxfeBNYg0sqdVdkqzNsiUmoO+e0+riNR6R5tdPTVKPyHr9D2YBvDTdKOyoa6qqHum8JV+HvpyRoPShuf2C98QkrUSbtjXhTnHRVhKIAs5YdYcykonsh8lYSMLzgytayj1dFYNouyQoH6Np7dqoYZfYJilOvEBb1T3amlgsqEkxzlt69LGUJyMDE8bnsUkQ3HgBGMY/s8XIHCqni/REtKqJhCjUTsmt42ql24q4VlrjMPUlIuYVWAXDxZuXsP0g3KLW6IU1zlIEbZtr1t6P076wJEOlfYIJMgilRRhhsuyfiD4RAoBO3cwIrG6TsAYxl4A0CQzXH/0BLQL+S2VJCHLoEzByBUkaWIR2Fo4R9o7xMi2AOPTg0K1bWqq9rct7CRo4FcCLlLGrZQzKTBDevj7w4Q43nwdbLNbUwkrYux9QFe0MekYQTdYv3YQxCvMhnRsE4zbcBYknWWgq3IQYNCo36x86/5trmSzc3SSFk6NUvYN0cgbfeXyFeSIME/Ai74syX0aLsa7QWFoxiAmqxyXhWefMbZZp+sh/1ZEMGWmRxD68f8tZgbmXdYZmqT61OC1AqAdbFuKt6p4Om2xD6V0QGHg4E5wqyVBMm+leIGwmMgc7q4U0MzGcBhDj5Il3nKWKdaS11qPi5gCCjONpWsOvFNxM4oNZ3Gq9AX1l4efi7oUY1YHVvP1CLIDCJYCp+EBg0rH4ZPus1FjnwjCNTBj7Kjtil1p25dRbWTODGZkMisp8ghpbCxqZqrjWNzdlZqbq9DJcFLmhbCwlCkp38RMu629RbEIyEgLd0EJaueVeh7BmYh+KqMpKB/Ty4vHTTsU5DYj9xEf6p4lKeR88+Iid9crrq+Ofv0+cYcMO+sjZh3viFkfuvEWhHx1APji/MYiu7LgwZOP304+d7BT5PmDqvx6xRySBWh4Y8YamiUxaD56/NrDDdxkDIAKJz3nz5fofdkD5v48eTm5IdPwE4+nJ9c81UmV6BNEX3iK49UFKtpjFcMg7tF/kBGHRGuYIhVFA3vyPjK6Zbw+YgPwvejOH7sgQhf/DresykSmZZODEsJXxx5s+HRwdw7mnmjwf5gPjoaDkfubLY/O9z1/Jnrj3ePdsfebG882PfHbHhEvgrme0eHg7E3GHGf4enJ6fvzjSMpxCi4S4tX3Snm5DwMneGhM3L4nJxDLpQeKVA9uTp9f/Ez9qe0VLYg5ys+fn/xsamkUur8/95cnZzenJ+Zy2d/XQHDnWd9MvZv0buTJAH8pdDQTx9vri6+/3xz7nwPyFCbmaft+/efkNFA+7Vyi3hJJT58+qH2LYzvMvx2dgVTd+i9c/7xZ1rE61/efwZ0lqP4dOUopfgKfVe6kjPJkArvO3exq17mhKVB7KOrEyTTZcDtPegZxZZ4KEAec5nWLppDlhX8BuSUS7kSfTMei+gCC83jZeBZnPH2sKkstnikAzl7GFr2UEdASTSz7hlIQ2TZCFy1Odjc6NXhvQt1SgllPSY07OJ5GCAB/+PiWsWEJTg6qzQ8tohFccmPTx+Ffh5OJ0BDZQgoCsvnNV6rOgADEBWLWFrEGfEO5I4cqW5OJjfrT9ZAF2LkUvHiXTThVIdasU0RNBQrAe9HhE5KtaHAnYksoMY/y7HRE/B3JR5aq/13xvGWg4QCishchvNWQqHMwb18Q/EY31JoAA5B8b61eKoTD2VU3sCbHT1W6EU3h/yOMcKVVaK2SjhuFC/8upjhlrDaDddvg1NDtfEV531Mn0DbQvlxstv8vbBLNRZZIQ64iAOTb1h+0con4wdU9wrYdA0FdDStBs6tOwwj3MAl9NXY+vJthSLgma81StR060yQN7Qq+isPT6lEFrSYc+YAqhMMvYzU5pHbgn6gSCeJTDH5YsR60T+0UZ9XoNXaEz7qnAim2WpZxJ4TU6EZFVHoYjQv2pmaKrWSkemClirsgaJIq9GjFeqvxPA50h1SsdJUpA3UNQDmUrTWsKDW2uvOP2lh1FzCEKHJhV6IB6FWEUksvJiMtKb4P7L22QZwGeKGa2NtnAq6hd1ZFocgvGkKt1GBMUyCFG2KcgE9T7ZUHmd9bVQxmf5wOTYuvYE9/39O0DCB5u8KFi51ZdViZFSatQJm7bkahKyen23xUJAq/Q3IhusmTetTDOoEUUd3QWhRXHSWUJyREPgAhGRFlsFjKwRcwhimImj40eeypfU3HgU1oT/SxsAZ3TG6wOELSAtoLUCKQ06RL/CazI+3x5uiZTEUFYqP/gT+/09ZL2WWE/FXX0khttdWs6evQbkEwu7AlVFOni/LQKtSBX2NZI70q9R8TabCUnUorRQ9S9eti2duKuhWZEYRdmv/R5C8g78dvTIJfG7qLQAxKxKfeIvhaynwOVAvO+U0lV70BtvZ70aTNksZQnMVowKFy5ccBwN/reeyzVLUUPrRD2GJXUILXhzEqu4yCgosm6AzWOUJrHJG+ul43dgEqLWMHxj30AAheKCcIFFdxhYIWXYnB6mDTLN9mIDH5yTWzGmcW9VpUp9pv+8mCYVE90Vr9ZwFuMVLnDD7HSSAKmRyI1GDsxOx8idJcoHDkZMrD10X9FOftt5WV/Pu9VN5fII7+OBPZIWYuYP7jpg1w0jUmIswjyk67n2L+3iEV69jXBK+8ckMtMpSYX6CeuKXNGZ1KwipYKveXr3YK7BWa6odf/8g8Xe6xQ+sPy7wkAXSHjo2vqAjfoUZHvNEGLDY6Wl5d9Yj3nQr9H1KDNIw6o0BemvEvg2kWY5e21nha9gRzoUjxevgFZ6imQi1802n9i+fZJQdb4I70LlzvYh+fpDZi+j4QumrtAmN9SbfyoVGOwh8P5PQFc75AvX1dEd6E1Sz4tqZUK2KCjHdqpTiLgvtkK3aZAGepZveA3ps4/Tm8Srytws5Wnhc4SsqThLGddAh7gm424fkmmCPLLU6PHjoiKgmAaw4DbRlUnI178YGtFicfa3TY5MYp50jKt38VTKkmUJ0wq7htiLgzeI45KFmx1Y8+xWU3IpRVDQz3cqBm/IsLVwPDDDHw4rJ5CtzlIK4HVk0/ddVjAG3GzSuRD/hduTaVk+KPm7mBYHIEaQ0D8Ii/N9JMaK/IykUaoHHFgbECqZwz5645VOoUmzugpRFEqh4JVyB2iv3q/qKBg0/jRGJUilG5yHvnzAa+u3J3rq6capz85QwEb/1M9ZWorcMGUHKIiiNPEPDL6WuaEWr5QzdeGoqDYGRMpZUTBDxUgQaFhN8TV8zlj8yBgKPaO+FqPazaKqUfaSjE5tSlitPAweNPKyjrVKxQBXMkGClQ+kVsMKeXeVIert2GMNeVYwMvJrYW8+yYK9A316JqS+vgzQ01LNykqrxqB5vo3HKzuXVp5tz2m+085wfz/8d3YLP0jmHniuf4ZG+PF5l5clQ/gXezJPhuP42Wjx6ylsRSAKURcrrYRwnSgH0FgGtAF6aKm9Fbh6oAlx5EagVwvhOLwsv7twly9yMDvCVH2DhUxdTu5B1ihIiKZ8yQIowdyPQgDKqr34mXxHMBhSrr0/qB/QJyZnwUjM3Y2qOpekWYHwU8fw1+QL9KFnCUqqpFMLImCR0n5pGmLhZ9hinvvoKBwPdBR6s9WJtAbTICue6VmARZ3n1HcaiK+9Shi9g1OSQjzyG4WYqHPjs9GnnbnYPFecuuoGV94ZxPLpphMVXEZ4XxEMB2keARRzdxQAXXJggosOR6woUXbyoBkR3zhySfR3uO0/dR43SF+aBwqP/SGFWYmPDI24mhdHDFg7RsSDKdYFWDcdlKLxxr063Tnw/QH7IzwADrSi27XDcw3hpD6RIABxgO21eZPBFlzRyxZIhQtalGw8/IzGRQ9JCBdSscWjuRFl9QlV086eiTxXloG3526YsSxmqzTCb7+puQOpOBvHiQ3fdgSpuMp4AabLpZwcm/J+dLyf9/3D7v92Kv4P+kXP7pjudZm8mQham8dRHzbOfmLQodSFqOtQc0/iRvGK5tTUiyHakZR0ZAYyX8/5j6xm/vlSlNX1gNCj7Lo1XSWdY8AJcLCP5/abhP+udvKDYA3sW1EE6iMuPa5MI6UZCSgdRL00DH2TiluE3rahgJfS53GmzVRD6IvhJ458VrskJhwOUo846JVn5nIaFdF2x0tNRZ9/RWpEnoDvl23KvdrQjEHpdOwOauySZWES1qOAQIUD1WhHLw9jTCtYLSaKnZiZZU0US9HqVbjtxKaCmSF0cpWWoDh6TL8hJMTTDAii0ugH+YoyGugpDEnU1M03RbQFYXrx9bkUtiueQvbuUhIUHUYnEK+KE1dcnB/mbcXz49T1nfubJUfU28FCJ75HBngCD/bwOWNReK8i09i4bAMjljKY9Q4KFdMr9REWhRdayiZAwFU0e1zaI1l2xucq3OonrVLMmVtpo32T1jabUrG40fecoBds3W2O15g1X3XQNyNkM+mInyh1Y7D4SXhyBYDzSTOXQ+tpS4U9c0vkBqlxEaKj+SDjX5YIJFUEpCMPHi166NiGm1BUdEnDR5mTQSUu9tMcl42tFMv5BSMZ4GmsI/xsdas4CbJtLsLXGtSavqMxPhZSLbQ2oOekNBmnQAenR4eJjS2tY8ool76SYae31rBE0NS74E8qujiK8OiS9bjR7XvmyrHsuBF9yBeGoRyjtaTCIIkeI+Rt1AdrBx4/XRLJveDXo7iehHdBMcDJaDxTSyXWxFsBgqTMqdBN/JoUNQYyjLu36JLzwVKpqqp2Sg4ACKM0iCjt9qebmUYhkUV6+q5dWSGBRWr6rlzZhINYs8PilkmcIFEAqdJL9JaOwaqFJa60KjfKT1ChpLGRbUnbcdrXUds/iZp1utVPz1qRAxrLYW6vWRVNNXC7RlT7wZhqggrMkK7WRoqZ5CZqmtgc3GaixYuM4q7tcLhknD9VBCbvAZWkX+BDHySajMtdsHJZGLnBMFUpTQz8zCcCaTZSljhppdAO9fCbt9kfQbjdDi2qtxjk1kxAcZkmPanOrkAgsrdAWY3Eo9u5yOCZcq1rIRG4DUWS729TAx/e/nK5pAIsYGkDr0Jmbu2dBqmK7FnxcrwUEoVJBxCIrRW8LWaiUGrUTgCS74RH9ZopLhS656cTi7GG8v7+7360YX4tEJ2vj9OaKmKpRy2KMldlWK17yET+X46+Xv23yRuoCcEX+aZtJw2zUgdWk5vr0GhhIa2uXJtaiyd0Nrd0aFd8m0fj1y9giGypDVaTql4agSF4EoHaJ06pQklaKotdEHKV4g+5adNBnWzeiab3Qh1N6g6pGt+KUIc8PP4XEm+1KH5nqivGAXvG8e4XJirwFIt27THb2Dh5PLi961i9sdk253ZWfZ0Hm8WymUtp5ZDNsEmAmKkoNiMfMof81yB3QMe6hiEgVZ3+Ax6Kckq8PVPWMsrE2lOc1/iz6FMAp5X4qblFqTCCAq0pcB4dWLbA6vsd46EpckHAKJgl+kwe0at+l9/e44uqrFbxLVliqOIVUKwDbKY+9OOSh2Xq2Ni0paWPNn6Vr/RhIZLVQkaaft7+jHjOpFcYsWrycTFzLI7YXTBi65OE4NdFi4t6hT1SNJ68tVzGKTmUIteUrP3aKSsclFqrL6j66QW4VpWyXQtxVk2s98xSvI5ELHxyQNjqNmAiNkh2k0y2D3Af2UD3YyJ16sskbXui8cgrUNN4Mtj8/dKz4P59BpnpKxCrws55kZwABPANA8/cnWtpIMV7Us92QTili/DLGKsU1z/VLt9s+KA9jUTrozJ7s7Y2Ouu3HLERk3HExfTWcTmRv0wIBqyqbUx7GaSqY0rkIBwVLaaYvgA3vVHs92d8SHiVRKOUDxeiKF3fk/Au3n36hQzH4C9t5flGsBbg6LcRLnktA8Zui5YgOoldQsfhQvsbae2g6C3CBeFyxwflQlsU9AbI4LGgHUcXkKi0BJaIlsWA9yZnw2kf17o0Zy6rh2caeijBGHmJOdRBbv1ydn1G0wW0thEMLgFRnqpBw5Isg+4NYiHtBnFx68wYdmdX49BLWTxQ2jy7WYgeVbcjqLxXspz6JyBXrbZhz8841w0fL5kLjwisyEheUtTjNJh2klrSrj0szZEVcMsGFpykQcU0d8dd4ZkuXOmkbyrSEaKsTOzTL/SBqOOBt2u3Kjud1eeKEjhxKKXp0bUZRrR0zudEb8TEXbqdbxXqiRA6lPnAo74XjhcyNVkllIxgG/51kVFDhAYgA5RFHihmHFlferBkuNcO72yyQJVDw6nhB1LXcHINmc7va4Pmnd5Y8dAvt4aHE07dvcRDMXVpiZNIJhFkmRa4OToWKHB9lg5xOZWRs92EKIg0vXxM8GWUh0Q45AadjkxYoXtgvbGYPrfN5tcmULQGS0OSDG4R0Rlpk/8XDkHcp7ND5KiSihfTKrqEKuTcBtaUj8iQM+cwATqukZ2GfSO7q/sk6rmn54msIJ7820BcdPzhLatgTQsokkbPThA/KNq/JpoYh1MUGs/iQx8IU0DEdpTHsa2O+CC3kR4cCxhTN1wbxW8/QBuo3etIBE7TkkV0BgEaA8RjMdcupnEVoJiI6yLIQkLwzHFhvrPGg21RaX9QqYYDNEGQLB8m5g2qMw7cSrOZdhR7DG9Kp1Agwfu9QceRA4f9/M01B8aNnDGPADSQ/Rao3k4KHld3Np9PsbWc6vX47ndrw2O38n2N49bf/1TVqoXKYJsND6N5lE+j84oePn67OT+sqRre2jfmQ15F1gxsHRqpdLaS7visdKaugilI8+OOW82FeBp2IXM2qcF/zPsNRyBgRMSA6Lt3GcInbAGWaTLdW+bx/iJuBpy2bkMkRpROTYl+Zk9ifIrtaM+iKcWU5W/ZUUJgkMqeOQ4AxV+fT6ZfOl/+cTm8pBOOWAjDE3OuLqjW1dm1fN8BCkHMqERW1GBZtMIbulY6+IOBJxtu6LTowHC5NgCY5VYGS3na6Riaj1tADZ75/a+ZM9ZE9Yq6ulI8M6plQg4Wtff3yjX39YuqrBja6fJNX0YagMOe3V02iddvim5gCbS5+7KiRIURxxBNA6OpVlc1mLM03kQJQIGn0JnPmg/TCAF8KGprU5UhM5oITIUnOjDjYG3GDJqWmLu1WYtYosMtnJNkW5AXWKk5Z7cSrFp7VKMoWgW4GKYT3KZS/loLFCeuvecPU6uFhValPw+3nZtRuFo2IOwOOwZCV3IYNQ1bvicwoUD8C9RGr90RYEyIMPgtLH2Lbhd88LpOGTwdSxJ6S9W+7uKewC/rSbW+NY1nG76e8E7EoWG+yQeVmoCushkD1l+tPH88Iq5r4TqnlZplxEfFaWkX8oEMytLAiKK6VUhX7qS7YNg+lRQrrFXvGXL0BcIEU7TbQEPAfXBAn8DUKRzfF4VNzNY4iuJsVXKEsZkWDldxlhoGKRjYbaAkusgY9v3nDq6NRQhn7y7cio2y7YcgVi9jbiTVs024yswOqtBLL9hrdSGqnFDow0YfQUk2YOSedNdCcbn2W+I5NWs96Dy8mLqt7dFCxsqarwWB2YD1zCH4Z3L7wEy51PJTHXGpMV0Ps3sYIb8iHZ1xo2E4iMpe5S+A24ngwvJKgEiS2jbMDr2rRjTdl2tDK78y0MXnu/2far2baRNOFnVc/AaNhWGH8LNPL1TezcDE0UmS8pdpzV3eLXLAs8jQhe6GOzfUodI4HOIkVYuI64s2q8jh59OPxY3WjwcBCZ92auvIacNwya3W9yk5TTcXqZoNRb7jZyIwpTps63GmRNe46vRhhTOnl+HI8uq3pXAgTvdaGFlunZ2GQOQ8irPu/OnqjSi6nqn0GnQg4PlwF0aLJeAYlbA/lubDTbXO+3YGKApj7pto/n4NTYFvWZtDK8pjuhEeK0iHD6LGFrvq1eozmuukJV81rjOe68atFCjCtSdVxhDPcxDYi8IAmKmK2cdytjET3GnCtfMka+ViT06q90dImzfknDVDsR/7abGrpfJ/G9yy6DIoTkcLzDAt/xTLGHakm8bUuCptB1uDxbTc808Zo4ubFBtmvTJXnM9nMxrSBv7hd35JDBca0DCIM+mo2OAtX7Ic4vl8lbTpGEzvd1CJegLcCyGai8nqQrFdDZef3QRh2uq3618aQaYNO3bHFp62SrKZQhAZpe7q1YGEYG0VsGfcxeU24iFpRBIxMhoYyktHNMQYEk4E+F6ErL1rUAQ80UFLpP+uRMC92bQxVc5X7aORIhEClszVlHkNnNZl/u+otDt1N9n+Zj1G9csF9NG/LDfVxTUQvojQk7PCaY5dutuRdy9MK/NQQtm4byW9rLMbAtHNMGCl4hGJXKdI0kmnld5uOS1MRFv9/8Ixk5sniYI8e+UQXLA3XzKxdAm4g9iUQPkfZKknoEjERDaXdNyvHVFzlZGzyHwajIkEhTDWJMaJWOURQzbZhIDf19MCfr8+vMPZNnnNq3cv17pVTCd/U/eXJ9fUvn67ONute9oons0maVtFFfsWI1hqaAnJRbGczAAm5GsoUYbD/WNzjpFj2ig5D7kcv0x6CUI5Xn2AOvfDp98C94WA43Hx/NkPPABgTijaAiC4sUhe38SRlowarEEKtJUEOW1XJDfVIE9K/ckJNxzv/ORMSPKRhDnits00X7aTMEWmnm6Wm2mr3WrDlVQZjVIC+cSwS3L0WyrXxWLqbGPIMbFVJ2yu2NI/osBuVs3ZeMexuLtXXeCpPbCZYKZo+XDx4UdxZWOJco4glGvJksDrHxwrd/gbp4x1d0SSeFekjU6WSf6T8UQ2YfPUpDXmjDozzSwmg28at3/umZtacdu79NyVIG50eaVadtsSpBIpLU9Hmkm7WQbOXmj3LHqH5ivhpDxM5+C7dD6uJcngiIxd5zih4GvBtjjmEuQNlxmCrTlejwWhcQzquirU41LR0noYsHkUUuh+klZOk8h/DTRIGGGqHuAzfxZmt5tMq0uSnXaJbG2LLTbhPG6UkrYEAo8KFGV1Le2HA/Ar8fJZ5aZDkaEYq8l/Lw+DK5WUZXV5m0o/pOjOhVjvlUQTHnDidLgqA4rY3vzOWwBvQpls7pmTF0lkwMRw/6RphokZm6bPrGrLsg8w99ynBvgqV6dYjhUbVwqXa8+/zDyLaV1mi5oEW6SPHRGWrCV5F0l4zfiNFEvvDeGSr5QgLL9CXCNKQ7V4Zp7mHPmad6qt5sKpNGPZgt30rSVNylTSIs1nlwQkHEzMasPONKddxOSZYz4npFIae1FgpjzaUV1XAbMibVwiDZZBP9gfighPxpx3T15jEiOE3+J3XOI7LPL5VM5VQowRVA24h2YNq+bKNcvMa85bi0TEjegUFsGRHC8YyYWdbJfLzblqpFgtsQGAjCju0ShNzo5oBtFuVo/jBKl+4fJU2QObAW7Ui9IlgCrtO1+Y3bFlvLfMVWxhATP9piVzeLEJZGBgbZtRkiDSINRjG37Mcs5vNjJfPotGeBtyXXpsg7zwuWLnX311cXd84p59+uvxwfnN+tsZnsXS/dga9+kr0169Et/ca1dEX7rL1C/ynSW04jd4JglWDd7HVy1iAudG7uFaBQ1djUpEygazcLJj1bETR47uX/iJepbpwSXTReqRLcjC+3beta9GyEFQbRMvKoGqOz7aJKFdImz2cfE8D6fPj/z7gbxg19xvI0fDIWNNomp1EMjW01CnX+Aa+LV5PQx5MP1HTUE7FsalN/QNrQzkrmnplms32/1cEf5UTWR/EVZ9h1WXQe5W9p3nu8gDapLq0VeOGEWxF7QllBqV0rK9yqYs66zzqdS+/EVdELMKWjEoQipPBbsSPX2MZM8JQDLs2O+n8/6+MGHgFdvGe1LnL+1rNkxfl7aYApo1gwuMTGiCyPlJDJaZt0RjfDN/XwbjauDH64pXLorC+vzt0sxDE+bHGIFsg5xMH1ovTkqDEhqHIl4pWvnXxnNOtiD1aMebUbCv6qgDNKp4oMS7H3ygvTLfeYSP97J/F8TcMO/2dGMBc95pqclBxfPlZ/PpD+tLGEExHXRx+A4oefEGijEmvNAfovVIDjTE/rnGcr4x4VmAkNgSeO2a+OKaKE3tWZtkeTiGkE0OyluoxeTpTKu6QtB4CuurAL0ozPKiu46ZdO1j6+rCdtsisZrs/NujTRa5YEEc0p5zJQA9WCT8c0hRWZrr/0Rw4ttmR3G+ZwFx38FQO6/694y4Xs6UfvqXMq1e/PL417PB3OQu8lpgVSbI1rkuuAnn/Q82yo0SlaqlCGhTh5vDUwi22thFzIKta8RWaDrGZ0sK7/kiJamZde+dk+xK2zq96CluDuhF7ak4MLZWOSJ9QxsWJ84oiSRBPV4WX/5JR23E413UcTGrgOAXTpaD0fp+8M32R2YaHhmdPme2mdw9aojlO734WV9/4VkFs5e03x9wirt2yZtgzFc4pUmatHgIvVv2F4gVdU6em3epZmF4OqNrApn9JxIUWJocH4/0ePwLOHhgG9QXRnO6seMSDCoX5KSOaAa+5GXTr2Nr6zvpzHuQhs67YXZBhtgkuLHXoAm/ML1le3524EcPcwkUdHtOFn9SoPr6lMcakTPpUcBXKcx2JmaMm3LNiwnRgGsEdIAT8FWnDSvtwD5eFR+1NIxoOIm4YzCQI0aFBicq0KzjvwngGO77TlXkpy68EiGCu3dmpBGgjblcv8OziFZ3oi+oWFxEYbtk5DQPvXgBGnF7I4+ICUQSIIRODcA0S5GELcpY0LwPFuf8JsFOkEg88EbSSr4DlhuWRRnKM4SYuPQU6ENQGy4R6tWpVCl0tYCcxhsUayLT5/rtntV+bkKfD8+DLbFPCrmdbmPQSjVkSehXYWO4dbGUOIZ5zR6TT4anBxNPJO+fi4/mNxCX7+tPpj871zdX5yU/kpeJZQtN4xoz38ZSf7VkAnBnAPhwd0KYbUqZl2G+VC3nEef+mC3gqWEI5ILEVFQSwvqCg4Y0ALiZukTcEiAXnJj5MG0OBs1mx9QT5MiGRcp+PaMURQXToA9Kj6vBibCSrhoJfTLF1yo29/JA5nZRVtgvux++sn8v7wpi4Elmhm4RcmFmnekWbnA16H2nVLY7q9jSS5PgUiTd+w5ZlUjm8oasBIFU6Zdt1sdR4R+gX5AnlmHsyBR5ABv6G7kyQfMXZaydPfIPWeIy8hRDAOzGAWl6Bql6GTZCUZTF/j8xKvn4YdIwJ6nPgtVb2wnjlz0NchqIakSASKID0KCPoWWWDkvS0ixLaBJwFLDtlYlZeFje5U8+giTvio3516yUv9nctjFwDeX1rbWS9dRe3Ek45fJAknagu+Oal7ZrAUOYmkpj9gU4XqSz18uKMRAwFLHYS+EoddCFIgAluk3Hu4tNF5hm/t5qfiRc5TwV/F6sl9xCV3qHkqJktrm74zvoFpTmZTQpvEcQhvr+5ubQwfSdSMcqciO14xemb2jbGqsVGluJPGsK2t0VoBkc6h9+Wg7l8x4OukUYT8dfr2vAosAiT+h/v7BRU+xiJLV7GXHixRsQIAEoJjLMqxIqDg4pNvyNL6vl4eUFpIcaEnyL27Ud+nuJUiTPgt/Uo5cu0nTJgrvkoxvrbR8lQUtyEI3EBrbuS2YBCx/vXdUfFzNOiQqJWpWJfIQA06AEcMdUNXibgqafV6X7pH+IVTbfd9besaqqqnKcwfDA3pdDlrn6VsswZBa8VGTzmx3uSu05lX0lZ1L6++OHm/OqnbpvAZxiMvN+XyITcQyXJLOlf+U4nfyWq48h4qWY6yKMGFdpXknJdTOQcaSXvHTFuEv2iXEEia2PvbXK1dSuVxKUwY9s3LQvQDU2drI24TmZPCzCJ0hqZFWSKU9kKTRqOSqLEo8yUhWxFc3knhyGZk7yHpLxXq3+Lub8AF8oFhWdQqYmOlbFTMl+XsgMx+Ag6UHMxqZGqEgTTaUQZAylBIGhm+OEBpDWCArZhrnPpZvIGWHgXcx0PL6FV9kKp8RVHDuYBC/1jbXPyRrGnIgFoCQmOpY+ZuJ2FUkOaMv9WRndNNK6YAq/WRPEEWZOr3UbVeBcIdwOxei3StiiQVaw0k7ZNyVqJxw0j+V2ooNnO6loa8iLqIpZJilgKeErPklCKt4J0CaNFnDTZLEA12tBkESc1i4WQUpBCwNB3hHYlBRYpu0uBBe0Vha2iIH+cSCiKukp0C6W9kJYb9HNdL19nMG3Uwss1LYq2YmJh0lf8TfyYhVJfE+ObJqDOlSws1QrVuVS+m+eki37VKhQbVZwOPhzomrnCoaR74WuC15JtqKurRK1k7HHSE8YbDMOhd0+Rx53oitoNACuyeppghR8FflVSfxbwK+vrMDHvQ7OBScZOFBfHC4xHC62LoelW5j7ICICyR41bqnIOd2HR+pE2wDMZlfCg1tDwpY6Fty7bIzuP4BvveKoc7rn933w3c3Uli3W3Lo6Wu8V4a1sv/w+JcNgQ'
SWHUB_ACTIONS.update(json.loads(_swhub_zlib.decompress(
    _swhub_base64.b64decode(_swhub_action_bundle))))
swhub_panel = show_swhub_panel('colab_contribution')


## Advanced implementation (optional)

These code blocks are collapsed by default. The top Open control panel block already includes everything the panel needs, so you do not need to run the blocks below. Expand them to inspect the implementation.


In [ ]:
%%swhub_action prepare
# @title Register prepare (runs only from the panel)
# @title Validate CUDA GPU and install Colab dependencies
import hashlib, os, shutil, subprocess, sys, urllib.request
from pathlib import Path

if shutil.which("nvidia-smi") is None:
    raise RuntimeError(
        "This notebook requires a Colab NVIDIA GPU runtime. "
        "Choose Runtime > Change runtime type > T4 GPU and rerun."
    )

gpu_probe = subprocess.run(
    ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
    text=True, capture_output=True, check=False,
)
if gpu_probe.returncode != 0 or not gpu_probe.stdout.strip():
    raise RuntimeError(
        "This notebook requires a Colab NVIDIA GPU runtime. "
        "Choose Runtime -> Change runtime type -> T4 GPU and rerun."
    )
print("Assigned NVIDIA GPU:", gpu_probe.stdout.strip().splitlines()[0])

def install_dependencies(include_analysis=False):
    print("Installing the packages needed by this notebook...")
    print("Python:", sys.version.split()[0])
    try:
        subprocess.run(["apt-get", "update", "-qq"], check=True)
        # Ubuntu 24.04 uses the t64 name, Ubuntu 22.04 uses libzip4.
        zip_probe = subprocess.run(
            ["apt-cache", "show", "libzip4t64"],
            text=True, stdout=subprocess.PIPE, stderr=subprocess.DEVNULL,
            check=False,
        )
        zip_package = "libzip4t64" if zip_probe.returncode == 0 and zip_probe.stdout.strip() else "libzip4"
        subprocess.run(
            ["apt-get", "install", "-y", "-qq", "ca-certificates", "curl",
             "wget", "p7zip-full", "rsync", "unzip", "libgomp1", "libomp-dev",
             "openssh-client", zip_package],
            check=True,
        )
        packages = ["fastapi==0.162.2", "uvicorn==0.54.0", "websockets==17.4"]
        if include_analysis:
            packages.append("nest_asyncio==1.6.0")
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "-q", *packages],
            check=True,
        )
    except subprocess.CalledProcessError as exc:
        raise RuntimeError(
            "Package installation failed. Read the command error above and retry Start server. "
            "If the problem started after a Colab runtime update, see the notebook's Troubleshooting section."
        ) from exc


install_dependencies()

CLOUDFLARED_VERSION = "2026.10.7"
CLOUDFLARED_URL = f"https://github.com/cloudflare/cloudflared/releases/download/{CLOUDFLARED_VERSION}/cloudflared-linux-amd64"
CLOUDFLARED_SHA256 = "1f99fb518b83dec2ba8d870a7cc991d82a77e6196295a3d2754ba17ab9f91872"
cloudflared_path = Path("/content/cloudflared")
cloudflared_tmp = cloudflared_path.with_suffix(".tmp")

def file_sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as source:
        for chunk in iter(lambda: source.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

if not cloudflared_path.exists() or file_sha256(cloudflared_path) != CLOUDFLARED_SHA256:
    cloudflared_tmp.unlink(missing_ok=True)
    request = urllib.request.Request(CLOUDFLARED_URL, headers={"User-Agent": "SWHub-Colab/1.0"})
    try:
        with urllib.request.urlopen(request, timeout=120) as response, cloudflared_tmp.open("wb") as output:
            shutil.copyfileobj(response, output)
        actual = file_sha256(cloudflared_tmp)
        if actual != CLOUDFLARED_SHA256:
            raise RuntimeError(f"cloudflared SHA-256 mismatch: {actual}")
        os.replace(cloudflared_tmp, cloudflared_path)
    except Exception:
        cloudflared_tmp.unlink(missing_ok=True)
        raise
cloudflared_path.chmod(0o755)
version_output = subprocess.run([str(cloudflared_path), "--version"], text=True, capture_output=True, check=True).stdout
if CLOUDFLARED_VERSION not in version_output:
    raise RuntimeError(f"Expected cloudflared {CLOUDFLARED_VERSION}, got: {version_output}")
print(version_output.strip())


In [ ]:
%%swhub_action drive
# @title Register drive (runs only from the panel)
# @title Connect Google Drive and restore the persistent cache
from google.colab import drive
from pathlib import Path
import subprocess

running_service = globals().get("service")
if running_service is not None and running_service.poll() is None:
    raise RuntimeError("Stop the running contributor before restoring its cache.")

drive.mount("/content/drive")
drive_cache = Path("/content/drive/MyDrive/SWHub/KataGoContribution")
runtime_cache = Path("/content/katago-contribute")
drive_cache.mkdir(parents=True, exist_ok=True)
runtime_cache.mkdir(parents=True, exist_ok=True)

def sync_runtime_cache_to_drive():
    subprocess.run(
        [
            "rsync",
            "-a",
            "--delete-delay",
            "--delete-excluded",
            "--exclude=*.tmp",
            "--exclude=*.tmp.*",
            f"{runtime_cache}/",
            f"{drive_cache}/",
        ],
        check=True,
    )

subprocess.run(
    [
        "rsync",
        "-a",
        "--delete",
        "--delete-excluded",
        "--exclude=*.tmp",
        "--exclude=*.tmp.*",
        f"{drive_cache}/",
        f"{runtime_cache}/",
    ],
    check=True,
)
print("Restored persistent KataGo cache from:", drive_cache)


In [ ]:
%%swhub_action write_app
# @title Register write app (runs only from the panel)
# @title Write the local contribution service
# This cell writes a self-contained service into the Colab runtime.
%%writefile /content/colab_katago_contribute.py
"""KataGo v1.18 distributed-training contributor for SWHub on Colab.

This service runs directly in a Colab runtime and is exposed through a
Cloudflare Quick Tunnel. Keep the generated tunnel URL private.
"""

import asyncio
import hashlib
import hmac
import json
import os
import re
import shutil
import subprocess
import sys
import tempfile
import urllib.request
import zipfile
from pathlib import Path
from urllib.parse import urlparse



# Keep this self-contained helper identical in both Colab notebooks.
# CUDA 13-only images do not provide the CUDA 12 SONAMEs used by KataGo.
CUDA_RUNTIME_DIR = Path('/content/swhub-katago-cuda12-12.8-cudnn9.8')
CUDA_REQUIREMENTS = '''\
nvidia-cuda-runtime-cu12==12.8.90 --hash=sha256:adade8dcbd0edf427b7204d480d6066d33902cab2a4707dcfc48a2d0fd44ab90
nvidia-cublas-cu12==12.8.4.1 --hash=sha256:8ac4e771d5a348c551b2a426eda6193c19aa630236b418086020df5ba9667142
nvidia-cudnn-cu12==9.8.0.87 --hash=sha256:d6b02cd0e3e24aa31d0193a8c39fec239354360d7d81055edddb69f35d53a4c8
nvidia-cuda-nvrtc-cu12==12.8.93 --hash=sha256:a7756528852ef889772a84c6cd89d41dfa74667e24cca16bb31f8f061e3e9994
nvidia-nvjitlink-cu12==12.8.93 --hash=sha256:81ff63371a7ebd6e6451970684f916be2eab07321b73c9d244dc2b4da7f73b88
'''


def _configure_cuda_library_path(runtime_root=None):
    # A new child process must see these paths, changing the notebook's
    # environment does not change the loader paths of its running Python.
    runtime_root = CUDA_RUNTIME_DIR if runtime_root is None else runtime_root
    directories = [str(path.resolve()) for path in sorted(runtime_root.glob('nvidia/*/lib'))]
    directories.extend(part for part in os.environ.get('LD_LIBRARY_PATH', '').split(os.pathsep) if part)
    for entry in sys.path:
        if not entry:
            continue
        root = Path(entry)
        for component in ('cublas', 'cuda_runtime', 'cudnn', 'cuda_nvrtc', 'nvjitlink'):
            for directory in (root / 'nvidia' / component / 'lib', root / component / 'lib'):
                if directory.is_dir():
                    directories.append(str(directory.resolve()))
        for directory in (root / 'nvidia' / 'cu12' / 'lib', root / 'torch' / 'lib'):
            if directory.is_dir():
                directories.append(str(directory.resolve()))
    for pattern in ('cuda-12*/lib64', 'cuda-12*/targets/x86_64-linux/lib', 'cuda/lib64'):
        for directory in sorted(Path('/usr/local').glob(pattern)):
            if directory.is_dir():
                directories.append(str(directory.resolve()))
    if directories:
        os.environ['LD_LIBRARY_PATH'] = os.pathsep.join(dict.fromkeys(directories))


def _probe_cuda_runtime(allow_cudnn8=False, minimum_cuda=12080):
    # Probe the standalone loader in a fresh process, never Torch's privately
    # preloaded libraries. Include cuDNN's lazily loaded component libraries.
    probe = r'''
import ctypes, json, sys
cuda = ctypes.CDLL('libcudart.so.12')
version = ctypes.c_int()
if cuda.cudaRuntimeGetVersion(ctypes.byref(version)) != 0 or not int(sys.argv[2]) <= version.value < 13000:
    raise RuntimeError('A compatible CUDA 12 runtime is required')
for name in ('libcublas.so.12', 'libcublasLt.so.12', 'libnvrtc.so.12', 'libnvJitLink.so.12'):
    ctypes.CDLL(name)
errors = []
for major in ([9, 8] if sys.argv[1] == '1' else [9]):
    try:
        cudnn = ctypes.CDLL(f'libcudnn.so.{major}')
        cudnn.cudnnGetVersion.restype = ctypes.c_size_t
        value = cudnn.cudnnGetVersion()
        parts = (value // 10000, value % 10000 // 100, value % 100) if value >= 10000 else (value // 1000, value % 1000 // 100, value % 100)
        if parts[0] != major or parts < ((9, 8, 0) if major == 9 else (8, 9, 7)):
            raise RuntimeError(f'Incompatible cuDNN version {parts}')
        components = ('ops', 'cnn', 'adv', 'graph', 'heuristic', 'engines_precompiled', 'engines_runtime_compiled') if major == 9 else ('ops_infer', 'cnn_infer', 'adv_infer')
        for component in components:
            ctypes.CDLL(f'libcudnn_{component}.so.{major}')
        print(json.dumps(parts))
        break
    except (OSError, RuntimeError) as error:
        errors.append(str(error))
else:
    raise RuntimeError('; '.join(errors))
'''
    result = subprocess.run(
        [sys.executable, '-c', probe, '1' if allow_cudnn8 else '0', str(minimum_cuda)],
        text=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, timeout=60,
    )
    if result.returncode:
        raise RuntimeError(result.stderr.strip() or result.stdout.strip())
    return tuple(json.loads(result.stdout))


def _ensure_cuda_runtime(allow_cudnn8=False, minimum_cuda=12080):
    _configure_cuda_library_path()
    try:
        return _probe_cuda_runtime(allow_cudnn8, minimum_cuda)
    except RuntimeError as error:
        print('Standalone CUDA 12 library check failed:', error, flush=True)
    print('Installing compatible NVIDIA CUDA 12 libraries for KataGo (first setup may take several minutes)...', flush=True)
    # Isolate the pinned libraries from Colab's Torch/TensorFlow packages and
    # Drive model caches. Publish only after installation and loader validation.
    CUDA_RUNTIME_DIR.parent.mkdir(parents=True, exist_ok=True)
    staging = Path(tempfile.mkdtemp(prefix='swhub-cuda-install-', dir=CUDA_RUNTIME_DIR.parent))
    previous_path = os.environ.get('LD_LIBRARY_PATH')
    try:
        requirements = staging / 'requirements.txt'
        requirements.write_text(CUDA_REQUIREMENTS)
        subprocess.run(
            [sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check',
             '--no-input', '--no-deps', '--only-binary=:all:', '--require-hashes',
             '--target', str(staging), '-r', str(requirements)],
            check=True, timeout=1800,
        )
        _configure_cuda_library_path(staging)
        _probe_cuda_runtime(False, minimum_cuda)
        if CUDA_RUNTIME_DIR.exists():
            shutil.rmtree(CUDA_RUNTIME_DIR)
        staging.rename(CUDA_RUNTIME_DIR)
    except Exception as error:
        raise RuntimeError(f'Could not prepare KataGo CUDA 12 libraries: {error}') from error
    finally:
        if staging.exists():
            shutil.rmtree(staging)
        if previous_path is None:
            os.environ.pop('LD_LIBRARY_PATH', None)
        else:
            os.environ['LD_LIBRARY_PATH'] = previous_path
    _configure_cuda_library_path()
    version = _probe_cuda_runtime(False, minimum_cuda)
    print('KataGo CUDA 12 libraries ready; cuDNN ' + '.'.join(map(str, version)), flush=True)
    return version


APP_NAME = "Katago-Contribute"
_gpu_probe = subprocess.run(
    ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
    text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, check=False,
)
if _gpu_probe.returncode != 0 or not _gpu_probe.stdout.strip():
    raise RuntimeError("A usable Colab NVIDIA CUDA GPU is required.")
COLAB_GPU = _gpu_probe.stdout.strip().splitlines()[0]

COLAB_CUDNN = _ensure_cuda_runtime()
SESSION_TIMEOUT_HOURS = 4

KATAGO_RELEASE = "v1.18.2"
KATAGO_URL = (
    "https://github.com/lightvector/KataGo/releases/download/v1.18.2/"
    "katago-v1.18.2-cuda12.8-cudnn9.8.0-linux-x64.zip"
)
KATAGO_SHA256 = "2cb197fc9bc2050f29112abb5b83cdbad63936cb4605d6e199772aedc9806c02"

CACHE = Path("/content/katago-contribute")
KATAGO_DIR = CACHE / "katago_v1_18_2_cuda12_8_cudnn9"
KATAGO_ARCHIVE = KATAGO_DIR / "katago.zip"
KATAGO_BIN = KATAGO_DIR / "katago"
KATAGO_EXTRACTED_BIN = KATAGO_DIR / "squashfs-root" / "AppRun"
CONTRIBUTE_BASE_DIR = CACHE / "contribute"
HOME_DATA_DIR = CACHE / "home"
LOG_DIR = CACHE / "logs"
DRIVE_CACHE_ENV = "SWHUB_CONTRIBUTOR_DRIVE_CACHE"

# Colab runs KataGo from the local runtime and periodically mirrors this
# cache to Drive. KataGo finalizes model downloads with an atomic rename,
# so excluding temporary files keeps partial downloads out of Drive.
def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as source:
        for chunk in iter(lambda: source.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def _download(url: str, target: Path, expected_sha256: str) -> None:
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.exists() and target.stat().st_size > 0:
        if _sha256(target) == expected_sha256:
            return
        target.unlink()
    temporary = target.with_suffix(target.suffix + ".tmp")
    temporary.unlink(missing_ok=True)
    request = urllib.request.Request(
        url,
        headers={
            "User-Agent": "SWHub-KataGo-Colab-Contributor/1.0",
            "Accept": "*/*",
        },
    )
    try:
        with urllib.request.urlopen(request, timeout=120) as response:
            with temporary.open("wb") as output:
                shutil.copyfileobj(response, output)
    except Exception:
        temporary.unlink(missing_ok=True)
        subprocess.run(
            [
                "wget",
                "--tries=3",
                "--timeout=60",
                "--user-agent=SWHub-KataGo-Colab-Contributor/1.0",
                "-O",
                str(temporary),
                url,
            ],
            check=True,
        )
    if not temporary.exists() or temporary.stat().st_size == 0:
        raise RuntimeError(f"Download produced an empty file: {url}")
    actual_sha256 = _sha256(temporary)
    if actual_sha256 != expected_sha256:
        temporary.unlink(missing_ok=True)
        raise RuntimeError(
            f"Download checksum mismatch for {url}: {actual_sha256}"
        )
    temporary.replace(target)


def _sync_cache_to_drive() -> None:
    drive_cache_value = os.environ.get(DRIVE_CACHE_ENV, "").strip()
    if not drive_cache_value:
        raise RuntimeError(
            "Google Drive cache is not configured. Run the Drive restore cell first."
        )
    drive_cache = Path(drive_cache_value)
    if not drive_cache.is_absolute():
        raise RuntimeError("Google Drive cache path must be absolute.")
    drive_cache.mkdir(parents=True, exist_ok=True)
    CACHE.mkdir(parents=True, exist_ok=True)
    result = subprocess.run(
        [
            "rsync",
            "-a",
            "--delete-delay",
            "--delete-excluded",
            "--exclude=*.tmp",
            "--exclude=*.tmp.*",
            f"{CACHE}/",
            f"{drive_cache}/",
        ],
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        check=False,
    )
    if result.returncode != 0:
        raise RuntimeError("Google Drive cache sync failed:\n" + result.stdout)


def _command_output(
    arguments: list[str],
    cwd: Path | None = None,
    timeout: int = 120,
) -> tuple[int, str]:
    result = subprocess.run(
        arguments,
        cwd=cwd,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        check=False,
        timeout=timeout,
    )
    return result.returncode, result.stdout


def ensure_katago() -> Path:
    KATAGO_DIR.mkdir(parents=True, exist_ok=True)
    if not KATAGO_BIN.exists():
        _download(KATAGO_URL, KATAGO_ARCHIVE, KATAGO_SHA256)
        with zipfile.ZipFile(KATAGO_ARCHIVE) as archive:
            archive.extractall(KATAGO_DIR)
        KATAGO_ARCHIVE.unlink(missing_ok=True)
    if not KATAGO_BIN.exists():
        raise RuntimeError(f"KataGo archive did not contain {KATAGO_BIN}")
    KATAGO_BIN.chmod(0o755)

    code, output = _command_output([str(KATAGO_BIN), "version"])
    if code == 0 and KATAGO_RELEASE.removeprefix("v") in output:
        return KATAGO_BIN

    if not KATAGO_EXTRACTED_BIN.exists():
        code, extract_output = _command_output(
            [str(KATAGO_BIN), "--appimage-extract"],
            cwd=KATAGO_DIR,
        )
        if code != 0:
            raise RuntimeError(
                "KataGo AppImage extraction failed:\n" + extract_output
            )
    # Drive-restored caches can lose execute bits on the wrapped binary.
    (KATAGO_EXTRACTED_BIN.parent / "usr" / "bin" / "katago").chmod(0o755)
    KATAGO_EXTRACTED_BIN.chmod(0o755)
    code, output = _command_output([str(KATAGO_EXTRACTED_BIN), "version"])
    if code != 0 and "error while loading shared libraries" in output:
        _, dependencies = _command_output(
            ["ldd", str(KATAGO_EXTRACTED_BIN.parent / "usr" / "bin" / "katago")]
        )
        raise RuntimeError(
            "KataGo's CUDA 12 / cuDNN 9 libraries could not be loaded. "
            "PyTorch's cuDNN check alone does not validate this executable.\n"
            + output + "\nDependency check:\n" + dependencies
            + "\nLD_LIBRARY_PATH=" + os.environ.get("LD_LIBRARY_PATH", "")
            + "\nLibraries marked 'not found' must be installed for CUDA 12 "
            "and cuDNN 9.8 or newer (major 9) in this runtime."
        )
    if code != 0 or KATAGO_RELEASE.removeprefix("v") not in output:
        raise RuntimeError("KataGo v1.18 validation failed:\n" + output)
    return KATAGO_EXTRACTED_BIN


def _bool(value: object) -> str:
    return "true" if value is True else "false"


def _quoted(value: object) -> str:
    return json.dumps(str(value), ensure_ascii=False)


def _int_in_range(
    config: dict,
    key: str,
    default: int,
    minimum: int,
    maximum: int,
) -> int:
    try:
        value = int(config.get(key, default))
    except (TypeError, ValueError) as exc:
        raise ValueError(f"{key} must be a number") from exc
    if value < minimum or value > maximum:
        raise ValueError(f"{key} must be between {minimum} and {maximum}")
    return value


def _tri_state(config: dict, key: str) -> str:
    value = str(config.get(key, "auto")).lower()
    if value not in {"auto", "true", "false"}:
        raise ValueError(f"{key} must be auto, true, or false")
    return value


_PROTECTED_EXTRA_KEYS = {
    "cudadevicetouse",
    "cudausefp16",
    "cudausenhwc",
    "disablepredownloadloop",
    "homedatadir",
    "includeownership",
    "logdir",
    "loggamesasjson",
    "maxratingmatches",
    "maxsimultaneousgames",
    "mirroruseproxy",
    "modeldownloadmirrorbaseurl",
    "numnnserverthreadspermodel",
    "onlyplayratingmatches",
    "password",
    "proxybasicauthpassword",
    "proxybasicauthusername",
    "proxyhost",
    "proxyport",
    "reportperformanceevery",
    "serverurl",
    "taskrepfactor",
    "username",
    "warntaskunusedkeys",
    "watchongoinggameinfile",
    "watchongoinggameinfilename",
}


def _safe_extra_lines(raw: object) -> list[str]:
    raw_text = str(raw or "")
    if len(raw_text) > 16000:
        raise ValueError("Additional config must be 16,000 characters or fewer")
    lines: list[str] = []
    for line in raw_text.splitlines():
        stripped = line.strip()
        if not stripped or stripped.startswith("#"):
            lines.append(line)
            continue
        match = re.match(r"^([A-Za-z][A-Za-z0-9_]*)\s*=", stripped)
        if not match:
            raise ValueError(
                f"Invalid additional config line (expected key = value): {line}"
            )
        if match.group(1).lower() in _PROTECTED_EXTRA_KEYS:
            raise ValueError(
                f"{match.group(1)} is managed by SWHub and cannot be overridden"
            )
        lines.append(line)
    return lines


def build_config(config: dict) -> str:
    server_url = str(config.get("serverUrl", "")).strip()
    parsed_server_url = urlparse(server_url)
    if (
        parsed_server_url.scheme != "https"
        or not parsed_server_url.netloc
        or parsed_server_url.username is not None
        or parsed_server_url.password is not None
    ):
        raise ValueError("serverUrl must be a valid https:// URL")
    username = str(config.get("username", "")).strip()
    password = str(config.get("password", ""))
    if not username or not password:
        raise ValueError("username and password are required")

    proxy_host = str(config.get("proxyHost", "")).strip()
    proxy_username = str(config.get("proxyBasicAuthUsername", "")).strip()
    proxy_password = str(config.get("proxyBasicAuthPassword", ""))
    mirror_url = str(config.get("modelDownloadMirrorBaseUrl", "")).strip()
    if mirror_url:
        parsed_mirror_url = urlparse(mirror_url)
        if (
            parsed_mirror_url.scheme != "https"
            or not parsed_mirror_url.netloc
            or parsed_mirror_url.username is not None
            or parsed_mirror_url.password is not None
        ):
            raise ValueError("modelDownloadMirrorBaseUrl must be https://")
    watch_name = Path(
        str(config.get("watchOngoingGameInFileName") or "watchgame.txt")
    ).name
    max_games = _int_in_range(
        config, "maxSimultaneousGames", 8, 1, 128
    )
    max_rating = _int_in_range(config, "maxRatingMatches", 1, 0, 128)
    task_rep_factor = _int_in_range(config, "taskRepFactor", 4, 2, 16)
    report_performance_every = _int_in_range(
        config, "reportPerformanceEvery", 120, 1, 21600
    )
    nn_threads = _int_in_range(
        config, "numNNServerThreadsPerModel", 2, 1, 16
    )
    cuda_device = _int_in_range(config, "cudaDeviceToUse", 0, 0, 15)

    values = [
        f"serverUrl = {_quoted(server_url)}",
        f"username = {_quoted(username)}",
        f"password = {_quoted(password)}",
        f"maxSimultaneousGames = {max_games}",
        "logGamesAsJson = true",
        f"includeOwnership = {_bool(config.get('includeOwnership', False))}",
        "watchOngoingGameInFile = "
        + _bool(config.get("watchOngoingGameInFile", False)),
        f"watchOngoingGameInFileName = {_quoted(watch_name)}",
        "onlyPlayRatingMatches = "
        + _bool(config.get("onlyPlayRatingMatches", False)),
        f"maxRatingMatches = {max_rating}",
        "disablePredownloadLoop = "
        + _bool(config.get("disablePredownloadLoop", False)),
        f"taskRepFactor = {task_rep_factor}",
        f"reportPerformanceEvery = {report_performance_every}",
        "warnTaskUnusedKeys = "
        + _bool(config.get("warnTaskUnusedKeys", False)),
        f"numNNServerThreadsPerModel = {nn_threads}",
        f"cudaDeviceToUse = {cuda_device}",
        f"cudaUseFP16 = {_tri_state(config, 'cudaUseFP16')}",
        f"cudaUseNHWC = {_tri_state(config, 'cudaUseNHWC')}",
        f"homeDataDir = {_quoted(HOME_DATA_DIR)}",
        f"logDir = {_quoted(LOG_DIR)}",
    ]
    if proxy_host:
        proxy_port = _int_in_range(config, "proxyPort", 0, 1, 65535)
        values.extend(
            [
                f"proxyHost = {_quoted(proxy_host)}",
                f"proxyPort = {proxy_port}",
            ]
        )
        if proxy_username:
            values.extend(
                [
                    f"proxyBasicAuthUsername = {_quoted(proxy_username)}",
                    f"proxyBasicAuthPassword = {_quoted(proxy_password)}",
                ]
            )
    if mirror_url:
        values.extend(
            [
                f"modelDownloadMirrorBaseUrl = {_quoted(mirror_url)}",
                "mirrorUseProxy = "
                + _bool(config.get("mirrorUseProxy", True)),
            ]
        )
    values.extend(_safe_extra_lines(config.get("extraConfig")))
    return "\n".join(values) + "\n"


def create_app():
    from fastapi import FastAPI, WebSocket, WebSocketDisconnect

    web_app = FastAPI()
    cache_commit_lock = asyncio.Lock()
    contribution_session_lock = asyncio.Lock()

    @web_app.get("/")
    async def status():
        return {
            "ok": True,
            "app": APP_NAME,
            "katago": KATAGO_RELEASE,
            "gpu": COLAB_GPU,
            "protocol": "swhub-katago-contribution",
            "protocolVersion": 1,
            "websocket": "/contribute",
            "note": "Connect with the SWHub KataGo Contribution page.",
        }

    @web_app.websocket("/contribute")
    async def contribute(websocket: WebSocket):
        await websocket.accept()
        try:
            await asyncio.wait_for(contribution_session_lock.acquire(), timeout=0.1)
        except asyncio.TimeoutError:
            await websocket.send_text(json.dumps({"type": "error", "message": "A contributor session is already active on this runtime."}))
            await websocket.close(code=4429)
            return
        process: asyncio.subprocess.Process | None = None
        config_path: Path | None = None
        reader_tasks: list[asyncio.Task] = []
        uploaded_games = 0
        latest_games: dict[str, dict] = {}
        send_lock = asyncio.Lock()
        shutting_down = False
        paused = False
        sensitive_values: list[str] = []

        def redact(text: str) -> str:
            result = text
            for value in sensitive_values:
                if value:
                    result = result.replace(value, "[REDACTED]")
            return result

        async def send(event_type: str, **data) -> None:
            payload = {"type": event_type, **data}
            async with send_lock:
                await websocket.send_text(
                    json.dumps(payload, separators=(",", ":"))
                )

        async def write_command(command: str) -> None:
            if process is None or process.stdin is None:
                return
            process.stdin.write((command + "\n").encode())
            await process.stdin.drain()

        def close_input_after_cleanup(text: str) -> None:
            # KataGo leaves its control thread blocked in getline(cin) at exit.
            # EOF releases it so C++ stream cleanup cannot hang after games and
            # uploads are done. Keep stdin open until this final log so forcequit
            # remains available throughout graceful shutdown.
            if text.endswith("All cleaned up, quitting"):
                if process is not None and process.stdin is not None:
                    process.stdin.close()

        async def commit_cache() -> None:
            async with cache_commit_lock:
                try:
                    await asyncio.to_thread(_sync_cache_to_drive)
                except Exception as exc:
                    print(f"Google Drive cache sync failed: {exc}", flush=True)

        async def periodic_commit() -> None:
            while process is not None and process.returncode is None:
                await asyncio.sleep(10 * 60)
                await commit_cache()

        def finish_data_from_upload_log(
            log_line: str,
        ) -> tuple[str, dict] | None:
            match = re.search(
                r"\buploaded sgf\s+(\S+\.sgf)(?:\s|$)",
                log_line,
                flags=re.IGNORECASE,
            )
            if match is None:
                return None
            sgf_path = Path(match.group(1))
            finish_data: dict[str, object] = {"finished": True}
            try:
                sgf_text = sgf_path.read_text(
                    encoding="utf-8", errors="replace"
                )
            except OSError:
                return sgf_path.stem, finish_data
            result_match = re.search(r"RE\[([^\]]*)\]", sgf_text)
            if result_match is None:
                return sgf_path.stem, finish_data
            result = result_match.group(1).strip()
            if result:
                finish_data["result"] = result
                upper_result = result.upper()
                if upper_result.startswith("B+"):
                    finish_data["winner"] = "B"
                elif upper_result.startswith("W+"):
                    finish_data["winner"] = "W"
                finish_data["resigned"] = upper_result.endswith("+R")
            return sgf_path.stem, finish_data

        async def read_stdout() -> None:
            nonlocal uploaded_games
            assert process is not None and process.stdout is not None
            while True:
                line = await process.stdout.readline()
                if not line:
                    return
                raw_text = line.decode(errors="ignore").strip()
                close_input_after_cleanup(raw_text)
                text = redact(raw_text)
                if not text:
                    continue
                if text.startswith("{"):
                    try:
                        data = json.loads(text)
                        if isinstance(data, dict) and data.get("gameId"):
                            latest_games[str(data["gameId"])] = dict(data)
                            await send("game", data=data)
                            continue
                    except json.JSONDecodeError:
                        pass
                if "uploaded sgf" in text.lower():
                    finish = await asyncio.to_thread(
                        finish_data_from_upload_log, raw_text
                    )
                    if finish is not None:
                        game_id, finish_data = finish
                        latest = latest_games.pop(game_id, None)
                        if latest is not None:
                            finished = {**latest, **finish_data}
                            await send("game", data=finished)
                    uploaded_games += 1
                    await send(
                        "uploaded",
                        uploadedGames=uploaded_games,
                        message=(
                            f"Uploaded game {uploaded_games}"
                            + (f" · {finish[0]}" if finish is not None else "")
                        ),
                    )
                else:
                    await send("log", stream="stdout", message=text)

        async def read_stderr() -> None:
            assert process is not None and process.stderr is not None
            while True:
                line = await process.stderr.readline()
                if not line:
                    return
                raw_text = line.decode(errors="ignore").strip()
                close_input_after_cleanup(raw_text)
                text = redact(raw_text)
                if not text:
                    continue
                lower = text.lower()
                event_type = (
                    "error"
                    if "uncaught exception" in lower
                    or "server returned error" in lower
                    or "not status code 200 ok" in lower
                    else "log"
                )
                await send(event_type, stream="stderr", message=text)

        async def drain_output_readers() -> None:
            output_readers = reader_tasks[:2]
            if not output_readers:
                return
            _, pending = await asyncio.wait(output_readers, timeout=10)
            for task in pending:
                task.cancel()
            await asyncio.gather(*output_readers, return_exceptions=True)

        async def stop_process(force: bool) -> None:
            nonlocal shutting_down, paused
            if process is None or process.returncode is not None:
                return
            shutting_down = True
            try:
                if not force and paused:
                    await write_command("resume")
                    paused = False
                await write_command("forcequit" if force else "quit")
            except (BrokenPipeError, ConnectionResetError):
                pass
            try:
                await asyncio.wait_for(
                    process.wait(),
                    timeout=15 if force else 120,
                )
            except asyncio.TimeoutError:
                try:
                    process.terminate()
                except ProcessLookupError:
                    return
                try:
                    await asyncio.wait_for(process.wait(), timeout=10)
                except asyncio.TimeoutError:
                    try:
                        process.kill()
                    except ProcessLookupError:
                        return
                    await process.wait()

        try:
            await send(
                "hello",
                protocol="swhub-katago-contribution",
                protocolVersion=1,
                message=f"Colab {COLAB_GPU} contributor ready for KataGo {KATAGO_RELEASE}.",
            )
            raw = await asyncio.wait_for(websocket.receive_text(), timeout=60)
            try:
                request = json.loads(raw)
            except json.JSONDecodeError:
                await send("error", message="Start request must be valid JSON.")
                await websocket.close(code=4400)
                return
            if not isinstance(request, dict):
                await send("error", message="Start request must be a JSON object.")
                await websocket.close(code=4400)
                return
            if request.get("protocolVersion") != 1:
                await send(
                    "error",
                    message="Unsupported SWHub contribution protocol version.",
                )
                await websocket.close(code=4400)
                return
            expected_endpoint_username = os.environ.get(
                "SWHUB_CONTRIBUTOR_USERNAME", ""
            )
            expected_endpoint_password = os.environ.get(
                "SWHUB_CONTRIBUTOR_PASSWORD", ""
            )
            endpoint_auth = request.get("endpointAuth")
            if bool(expected_endpoint_username) != bool(expected_endpoint_password):
                await send(
                    "error",
                    message="Colab endpoint login is configured incompletely.",
                )
                await websocket.close(code=1011)
                return
            if expected_endpoint_username:
                endpoint_username = (
                    str(endpoint_auth.get("username", ""))
                    if isinstance(endpoint_auth, dict)
                    else ""
                )
                endpoint_password = (
                    str(endpoint_auth.get("password", ""))
                    if isinstance(endpoint_auth, dict)
                    else ""
                )
                if not (
                    hmac.compare_digest(
                        endpoint_username, expected_endpoint_username
                    )
                    and hmac.compare_digest(
                        endpoint_password, expected_endpoint_password
                    )
                ):
                    await send("error", message="Contributor login failed.")
                    await websocket.close(code=4401)
                    return
            if request.get("action") != "start" or not isinstance(
                request.get("config"), dict
            ):
                await send("error", message="First message must be a start request.")
                await websocket.close(code=4400)
                return
            sensitive_values.extend(
                [
                    str(request["config"].get("password", "")),
                    str(request["config"].get("proxyBasicAuthPassword", "")),
                    str(endpoint_auth.get("password", ""))
                    if isinstance(endpoint_auth, dict)
                    else "",
                ]
            )

            await send("connecting", message="Preparing KataGo v1.18.2 in Colab, mandatory contribution safety checks run before games begin…")
            katago = await asyncio.to_thread(ensure_katago)
            contribution_dirs = [
                CONTRIBUTE_BASE_DIR,
                HOME_DATA_DIR,
                LOG_DIR,
            ]
            for directory in contribution_dirs:
                directory.mkdir(parents=True, exist_ok=True)

            config_text = build_config(request["config"])
            descriptor, temporary_name = tempfile.mkstemp(
                prefix="swhub_contribute_",
                suffix=".cfg",
                dir="/tmp",
                text=True,
            )
            config_path = Path(temporary_name)
            with os.fdopen(descriptor, "w", encoding="utf-8") as output:
                output.write(config_text)
            config_path.chmod(0o600)

            arguments = [
                str(katago),
                "contribute",
                "-config",
                str(config_path),
                "-base-dir",
                str(CONTRIBUTE_BASE_DIR),
            ]
            process = await asyncio.create_subprocess_exec(
                *arguments,
                stdin=asyncio.subprocess.PIPE,
                stdout=asyncio.subprocess.PIPE,
                stderr=asyncio.subprocess.PIPE,
                limit=50 * 1024 * 1024,
            )
            await send(
                "started",
                message=(
                    f"KataGo {KATAGO_RELEASE} is contributing in Colab {COLAB_GPU}."
                ),
            )
            reader_tasks = [
                asyncio.create_task(read_stdout()),
                asyncio.create_task(read_stderr()),
                asyncio.create_task(periodic_commit()),
            ]

            process_wait = asyncio.create_task(process.wait())
            session_deadline = asyncio.get_running_loop().time() + SESSION_TIMEOUT_HOURS * 60 * 60
            while process.returncode is None:
                receive = asyncio.create_task(websocket.receive_text())
                done, _ = await asyncio.wait(
                    {receive, process_wait},
                    return_when=asyncio.FIRST_COMPLETED,
                    timeout=max(0, session_deadline - asyncio.get_running_loop().time()),
                )
                if not done or asyncio.get_running_loop().time() >= session_deadline:
                    receive.cancel()
                    await asyncio.gather(receive, return_exceptions=True)
                    await send("stopping", message=f"The {SESSION_TIMEOUT_HOURS:g}-hour contribution limit was reached. Stopping KataGo…")
                    await stop_process(force=True)
                    break
                if process_wait in done:
                    receive.cancel()
                    await asyncio.gather(receive, return_exceptions=True)
                    break
                raw = receive.result()
                try:
                    command_message = json.loads(raw)
                except json.JSONDecodeError:
                    await send("warning", message="Command must be valid JSON.")
                    continue
                if not isinstance(command_message, dict):
                    await send(
                        "warning",
                        message="Command must be a JSON object.",
                    )
                    continue
                command = command_message.get("action")
                if command == "pause":
                    await write_command("pause")
                    paused = True
                    await send("paused", paused=True, message="Contribution paused.")
                elif command == "resume":
                    await write_command("resume")
                    paused = False
                    await send(
                        "resumed", paused=False, message="Contribution resumed."
                    )
                elif command == "quit":
                    shutting_down = True
                    if paused:
                        await write_command("resume")
                        paused = False
                    await write_command("quit")
                    await send(
                        "stopping",
                        message=(
                            "KataGo is finishing active games and will not start "
                            "new ones."
                        ),
                    )
                elif command == "forcequit":
                    await send("stopping", message="Force-stopping KataGo…")
                    await stop_process(force=True)
                    break
                else:
                    await send(
                        "warning",
                        message=f"Unsupported contribution command: {command!r}",
                    )

            return_code = await process_wait
            await drain_output_readers()
            await send(
                "stopped",
                uploadedGames=uploaded_games,
                message=f"KataGo exited with code {return_code}.",
            )
        except WebSocketDisconnect:
            print("SWHub viewer disconnected, stopping KataGo.", flush=True)
        except asyncio.TimeoutError:
            try:
                await send("error", message="Timed out waiting for startup data.")
            except Exception:
                pass
        except Exception as exc:
            try:
                await send("error", message=f"Contributor failed: {exc}")
            except Exception:
                pass
            print(f"Contributor failed: {exc!r}", flush=True)
        finally:
            try:
                if process is not None and process.returncode is None:
                    await stop_process(force=True if not shutting_down else False)
                for task in reader_tasks:
                    task.cancel()
                if reader_tasks:
                    await asyncio.gather(*reader_tasks, return_exceptions=True)
                if config_path is not None:
                    config_path.unlink(missing_ok=True)
                if process is not None:
                    await commit_cache()
            finally:
                contribution_session_lock.release()

    return web_app


if __name__ == "__main__":
    if "--check-runtime" in sys.argv:
        print("Validated KataGo executable:", ensure_katago(), flush=True)
    else:
        import uvicorn
        uvicorn.run(create_app(), host="0.0.0.0", port=8765, log_level="info", ws="websockets")


In [ ]:
%%swhub_action start
# @title Register start (runs only from the panel)
# @title Start the contributor and print the SWHub WebSocket URL
import json, os, re, signal, socket, subprocess, sys, time
from pathlib import Path

drive_cache = globals().get("drive_cache")
if drive_cache is None or not Path(drive_cache).is_dir():
    raise RuntimeError("Click Start server to restore the Google Drive cache before starting.")
for process_name in ("service", "tunnel"):
    existing_process = globals().get(process_name)
    if existing_process is not None and existing_process.poll() is None:
        raise RuntimeError(f"{process_name.title()} is already running. Use Stop server before starting again.")
with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as port_probe:
    try:
        port_probe.bind(("127.0.0.1", 8765))
    except OSError as exc:
        raise RuntimeError("Port 8765 is already in use by an unmanaged service. Stop it or restart the runtime before starting.") from exc
service_environment = os.environ.copy()
service_environment["SWHUB_CONTRIBUTOR_DRIVE_CACHE"] = str(drive_cache)

# Validate the actual executable and its shared libraries before opening a tunnel.
print("Checking the KataGo CUDA runtime before starting the contributor...", flush=True)
subprocess.run(
    [sys.executable, "/content/colab_katago_contribute.py", "--check-runtime"],
    env=service_environment,
    check=True,
)

service_log = Path("/content/colab_katago_contribute.log")
tunnel_log = Path("/content/colab_cloudflared.log")
for path in (service_log, tunnel_log):
    path.unlink(missing_ok=True)

service_log_handle = service_log.open("w")
new_service = subprocess.Popen(
    [sys.executable, "/content/colab_katago_contribute.py"],
    stdout=service_log_handle, stderr=subprocess.STDOUT,
    start_new_session=True,
    env=service_environment,
)
service_log_handle.close()
print("Local contributor PID:", new_service.pid)
print("The service restores cached files first and downloads only missing KataGo files/models.")

# Wait until the local HTTP app is accepting connections before opening the tunnel.
import urllib.request
for _ in range(60):
    try:
        with urllib.request.urlopen("http://127.0.0.1:8765/", timeout=2) as response:
            status = json.load(response)
        if status.get("app") != "Katago-Contribute" or status.get("protocol") != "swhub-katago-contribution":
            raise RuntimeError(f"Unexpected service on port 8765: {status}")
        break
    except Exception:
        if new_service.poll() is not None:
            print(service_log.read_text(errors="replace")[-8000:])
            raise RuntimeError("Contributor service exited early.")
        time.sleep(1)
else:
    os.killpg(new_service.pid, signal.SIGTERM)
    raise RuntimeError("Contributor service did not start.")

tunnel_log_handle = tunnel_log.open("w")
try:
    new_tunnel = subprocess.Popen(
        ["/content/cloudflared", "tunnel", "--url", "http://127.0.0.1:8765"],
        stdout=tunnel_log_handle, stderr=subprocess.STDOUT,
        start_new_session=True,
    )
except Exception:
    os.killpg(new_service.pid, signal.SIGTERM)
    raise
finally:
    tunnel_log_handle.close()
print("Cloudflare tunnel PID:", new_tunnel.pid)
for _ in range(120):
    text = tunnel_log.read_text(errors="replace")
    match = re.search(r"https://[A-Za-z0-9-]+\.trycloudflare\.com", text)
    if match:
        base = match.group(0)
        print("\nKeep this URL private:", base)
        print("\nPaste this into SWHub's Contributor WebSocket endpoint field:")
        print(base.replace("https://", "wss://") + "/contribute")
        print("\nStatus:", base + "/")
        break
    if new_tunnel.poll() is not None:
        print(text[-8000:])
        os.killpg(new_service.pid, signal.SIGTERM)
        raise RuntimeError("Cloudflare tunnel exited early.")
    time.sleep(1)
else:
    os.killpg(new_tunnel.pid, signal.SIGTERM)
    os.killpg(new_service.pid, signal.SIGTERM)
    raise RuntimeError("Timed out waiting for a trycloudflare.com URL.")

service = new_service
tunnel = new_tunnel


In [ ]:
%%swhub_action stop
# @title Register stop (runs only from the panel)
# @title Stop the contributor before closing/restarting the runtime
import os, signal, subprocess
for name in ("tunnel", "service"):
    process = globals().get(name)
    if process is not None and process.poll() is None:
        os.killpg(process.pid, signal.SIGTERM)
        print("Stopping", name, process.pid)

service_process = globals().get("service")
if service_process is not None and service_process.poll() is None:
    try:
        service_process.wait(timeout=180)
    except subprocess.TimeoutExpired as exc:
        raise RuntimeError("Contributor did not stop, cache was not synced.") from exc

sync_cache = globals().get("sync_runtime_cache_to_drive")
if sync_cache is None:
    raise RuntimeError("The Google Drive cache must be restored before it can be saved.")
sync_cache()
print("Contributor stopped and the latest cache was saved to Google Drive.")
print("Use SWHub's Finish games & stop first so active games can upload.")


## Troubleshooting

- **No GPU:** select T4 GPU or another available NVIDIA GPU in Runtime settings, then click Start server again.
- **Already running:** click Stop server before starting again. Keep only one contribution session connected to this runtime.
- **The address stops working:** restart setup and use the newly printed address in SWHub. Colab connection addresses are temporary.
- **Package installation fails:** read the first error above the traceback, then click Start server again. If the same error continues, use **Runtime > Disconnect and delete runtime**, reconnect and reopen the panel and click Start server again.
- **A Colab update breaks setup:** open **Runtime > Change runtime type > Runtime Version** and try an available earlier version, such as `2026.07`, as a temporary workaround. Recheck the GPU selection, then click Start server again. Google keeps older versions for a limited time. [Runtime version help](https://research.google.com/colaboratory/runtime-version-faq.html)
- **CUDA libraries are missing:** setup normally repairs these automatically. The first repair downloads about 1.4 GB and can take several minutes. It keeps Colab's PyTorch installation intact.
